In [4]:
# Import the section_properties module
from steel_lib.section_properties import create_aisc_section_selector

# Create the AISC selector
try:
    aisc = create_aisc_section_selector('aisc-shapes-database-v16.0.xlsx')
    print("AISC database loaded successfully!")
    print(f"Sections: {aisc['database_info']['n_sections']}")
    print(f"Properties: {aisc['database_info']['properties_count']}")
except Exception as e:
    print(f"Error loading AISC database: {e}")

Loading AISC Shapes Database...
Loaded 2299 sections from AISC database
AISC database loaded successfully!
Sections: 2299
Properties: 69


In [2]:
# Test the fixed filtering function
results = aisc['select_by_properties']({
    "W": {"max": 10.0,"min": 8.0},
})
results = aisc['select_by_properties']({
    "designation": ["W14X53"],
})
# props = aisc['get_properties']('W14X53')
# results
# print(f"Found {results['count']} sections")
# if results['count'] > 0:
#     print("First few sections:")
#     for i in range(min(3, results['count'])):
#         designation = results['designations'][i]
#         d = results['d'][i] 
#         Ix = results['Ix'][i]
#         print(f"  {designation}: d={d:.1f}, Ix={Ix:.0f}")
results

{'indices': array([211], dtype=int32),
 'count': 1,
 'designations': array(['W14X53'], dtype='<U20'),
 'W': array([53.], dtype=float32),
 'A': array([15.6], dtype=float32),
 'd': array([13.9], dtype=float32),
 'ddet': array([13.875], dtype=float32),
 'Ht': array([0.], dtype=float32),
 'h': array([0.], dtype=float32),
 'OD': array([0.], dtype=float32),
 'bf': array([8.06], dtype=float32),
 'bfdet': array([8.], dtype=float32),
 'B': array([0.], dtype=float32),
 'b': array([0.], dtype=float32),
 'ID': array([0.], dtype=float32),
 'tw': array([0.37], dtype=float32),
 'twdet': array([0.375], dtype=float32),
 'tf': array([0.66], dtype=float32),
 'tfdet': array([0.6875], dtype=float32),
 't': array([0.], dtype=float32),
 'tnom': array([0.], dtype=float32),
 'kdes': array([1.25], dtype=float32),
 'kdet': array([1.5], dtype=float32),
 'k1': array([1.], dtype=float32),
 'x': array([0.], dtype=float32),
 'y': array([0.], dtype=float32),
 'eo': array([0.], dtype=float32),
 'xp': array([0.], dtype=

In [3]:
import numpy as np
from numba import jit,njit, prange
import time
import forallpeople as si
from steel_lib.aisc_14th import optional_reporting_handcalc
si.environment("default")

# A computationally intensive function in pure Python
def sum_squared_diff_python(x, y):
    """
    Calculates the sum of squared differences between two arrays.
    """
    result = 0
    for i in range(len(x)):
        result += (x[i] - y[i])**2
    return result

# The same function accelerated with Numba's @jit decorator
@optional_reporting_handcalc(config_object=None,key=None, detailed=False)
def sum_squared_diff_numba(x, y):
    """
    Calculates the sum of squared differences between two arrays using Numba.
    The @jit(nopython=True) decorator compiles this function to machine code.
    """
    result = 0 
    for i in prange(len(x)):
        result += (x[i]  - y[i]  )**2
    return result

# The same function accelerated with Numba's @jit decorator
@jit(nopython=True)
def sum_squared_diff_numba_1(x, y):
    """
    Calculates the sum of squared differences between two arrays using Numba.
    The @jit(nopython=True) decorator compiles this function to machine code.
    """
    result = 0 
    for i in prange(len(x)):
        result += (x[i]  - y[i]  )**2
    return result
# Create two large NumPy arrays with random data
array1 = np.random.rand(10_000_000)
array2 = np.random.rand(10_000_000)

In [4]:
%%timeit
result_python = sum_squared_diff_python(array1, array2)

3.34 s ± 104 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [5]:
# %%timeit
# result_numba = sum_squared_diff_numba(array1, array2)

In [6]:
%%timeit
result_numba_1 = sum_squared_diff_numba_1(array1, array2)

13.9 ms ± 600 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [1]:
%%time
import numpy as np
from numba import njit, prange

from steel_lib.aisc_14th import (
    block_shear,
    bolt_bearing,
    bolt_shear,
    flexural_14th,
    prying_action,
    shear_yielding_rupture,
    flexural_15th,
)

num_cases = 10000000
# --- Generate random data for bolt_bearing ---
F_u = np.random.uniform(58, 65, num_cases)
d_bolt = np.random.uniform(0.5, 1.5, num_cases)
t = np.random.uniform(0.25, 1.0, num_cases)
P_u = np.random.uniform(0, 100, num_cases)
V_u = np.random.uniform(50, 200, num_cases)
S_r = np.random.uniform(3, 6, num_cases)
N_r = np.random.randint(2, 8, num_cases)
S_c = np.random.uniform(3, 6, num_cases)
N_c = np.random.randint(1, 4, num_cases)
L_ev = np.random.uniform(1.5, 3.0, num_cases)
L_eh = np.random.uniform(1.5, 3.0, num_cases)
dv = np.random.uniform(0.5, 1.5, num_cases) + 0.0625
dh = np.random.uniform(0.5, 1.5, num_cases) + 0.0625
phi = np.full(num_cases, 0.75)
c = np.random.uniform(1.0, 2.0, num_cases)

# --- Generate random data for block_shear ---
F_y = np.random.uniform(36, 50, num_cases)
coped_pattern = np.array([0, 1, 2])
coped = np.random.choice(coped_pattern, size=num_cases)

# --- Generate random data for prying_action ---
F_nt = np.random.uniform(90, 113, num_cases)
F_nv_prying = np.random.uniform(54, 68, num_cases)
n_bolts = np.random.randint(4, 12, num_cases)
N_t = np.random.randint(2, 6, num_cases)
L = np.random.uniform(3, 6, num_cases)
ga = np.random.uniform(3, 5.5, num_cases)
prying_pattern = np.array([True, False])
prying = np.random.choice(prying_pattern, size=num_cases)

# --- Generate random data for bolt_shear ---
F_nv = np.random.uniform(54, 68, num_cases)
A_bolt = (d_bolt**2 / 4) * np.pi
N_shear_planes = np.random.randint(1, 3, num_cases)

# --- Generate random data for shear_yielding_rupture ---
d_b = d_bolt.copy()
n_members = np.random.randint(1, 3, num_cases)
coped_syr_pattern = np.array([0, 1, 2])
coped_syr = np.random.choice(coped_syr_pattern, size=num_cases)

# --- Generate random data for flexural_14th ---
l = np.random.uniform(2, 6, num_cases)
k_a = np.random.uniform(0.5, 1.5, num_cases)
d = np.random.uniform(3, 8, num_cases)
e_override = np.random.uniform(1, 3, num_cases)

# --- Generate random data for flexural_15th ---
MEMBER_TYPE_CODES = {'PL': 0, 'L': 1}
member_code_pattern = np.array([0, 1])
member_type_codes = np.random.choice(member_code_pattern, size=num_cases)
a = np.random.uniform(5, 15, num_cases)
E = np.full(num_cases, 29000)

@njit(parallel=True)
def test_bolt_strength_function(
    F_u, d_bolt, t, P_u, V_u, S_r, N_r, S_c, N_c, L_ev, L_eh, dv, dh, c,
    F_y, coped, n_members, coped_syr, d_b, l, k_a, d, e_override,
    F_nt, F_nv_prying, n_bolts, N_t, L, ga, prying,
    member_type_codes, a, E,
    phi,
    results_bearing, results_shear, results_block, results_syr, results_flexural, results_prying, results_flexural_15th,
    num_cases
):
    for i in prange(num_cases):
        n_c = 1 if i % 2 == 0 else N_c[i]

        results_bearing[i] = bolt_bearing(
            F_u=F_u[i],
            d_bolt=d_bolt[i],
            t=t[i],
            P_u=P_u[i],
            V_u=V_u[i],
            S_r=S_r[i],
            N_r=N_r[i],
            S_c=S_c[i],
            N_c=n_c,
            L_ev=L_ev[i],
            L_eh=L_eh[i],
            d_v=dv[i],
            d_h=dh[i],
            phi=phi[i],
            c=c[i]
)

        results_shear[i] = bolt_shear(
            F_nv=F_nv[i],
            A_bolt=A_bolt[i],
            N_shear_planes=N_shear_planes[i],
            phi=phi[i]
)

        results_block[i] = block_shear(
            P_u=P_u[i],
            V_u=V_u[i],
            F_y=F_y[i],
            F_u=F_u[i],
            t=t[i],
            N_r=N_r[i],
            S_r=S_r[i],
            N_c=n_c,
            S_c=S_c[i],
            L_ev=L_ev[i],
            L_eh=L_eh[i],
            d_v=dv[i],
            d_h=dh[i],
            phi=phi[i],
            coped=coped[i]
)

        syr_val = shear_yielding_rupture(
            N_r=N_r[i],
            S_r=S_r[i],
            L_ev=L_ev[i],
            t=t[i],
            d_b=d_b[i],
            F_y=F_y[i],
            F_u=F_u[i],
            phi=phi[i],
            n_members=n_members[i],
            coped=coped_syr[i]
)
        if syr_val is not None:
            results_syr[i] = syr_val
        else:
            results_syr[i] = np.nan

        results_flexural[i] = flexural_14th(
            l=l[i],
            k_a=k_a[i],
            L_eh=L_eh[i],
            N_c=n_c,
            S_c=S_c[i],
            t=t[i],
            N_r=N_r[i],
            S_r=S_r[i],
            d_b=d_b[i],
            d=d[i],
            F_y=F_y[i],
            F_u=F_u[i],
            e_override=e_override[i]
)

        prying_val = prying_action(
            t=t[i],
            F_u=F_u[i],
            F_nv=F_nv_prying[i],
            F_nt=F_nt[i],
            N_t=N_t[i],
            n_bolts=n_bolts[i],
            L=L[i],
            L_eh=L_eh[i],
            L_ev=L_ev[i],
            d_bolt=d_bolt[i],
            d_v=dv[i],
            S_r=S_r[i],
            ga=ga[i],
            V_u=V_u[i],
            P_u=P_u[i],
            prying=prying[i]
)
        if prying_val is not None:
            results_prying[i] = prying_val
        else:
            results_prying[i] = np.nan

        member_type_code = member_type_codes[i]
        member_type_str = 'PL' if member_type_code == 0 else 'L'

        flexural_15th_val = flexural_15th(
            member_type=member_type_str,
            a=a[i],
            t=t[i],
            F_y=F_y[i],
            E=E[i],
            d_b=d_b[i],
            L_ev=L_ev[i],
            F_u=F_u[i],
            N_r=N_r[i],
            S_r=S_r[i],
            d_bolt=d_bolt[i]
)
        if flexural_15th_val is not None:
            results_flexural_15th[i] = flexural_15th_val
        else:
            results_flexural_15th[i] = np.nan

    return (
        results_bearing,
        results_shear,
        results_block,
        results_syr,
        results_flexural,
        results_prying,
        results_flexural_15th,
    )

CPU times: total: 4.08 s
Wall time: 11.3 s


In [2]:
# Initialize result arrays for the test function
results_bearing = np.zeros(num_cases, dtype=np.float64)
results_shear = np.zeros(num_cases, dtype=np.float64)
results_block = np.zeros(num_cases, dtype=np.float64)
results_syr = np.zeros(num_cases, dtype=np.float64)
results_flexural = np.zeros(num_cases, dtype=np.float64)
results_prying = np.zeros(num_cases, dtype=np.float64)
results_flexural_15th = np.zeros(num_cases, dtype=np.float64)

In [5]:
%%timeit
test_bolt_strength_function(
    F_u, d_bolt, t, P_u, V_u, S_r, N_r, S_c, N_c, L_ev, L_eh, dv, dh, c,
    F_y, coped, n_members, coped_syr, d_b, l, k_a, d, e_override,
    F_nt, F_nv_prying, n_bolts, N_t, L, ga, prying,
    member_type_codes, a, E,
    phi,
    results_bearing, results_shear, results_block, results_syr, results_flexural, results_prying, results_flexural_15th,
    num_cases
)

453 ms ± 39.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [10]:
from math import sin,cos,pi
from handcalcs.decorator import handcalc
import forallpeople as si
from numba import njit
from steel_lib.aisc_14th import bolt_bearing, block_shear, shear_yielding_rupture, tensile_yielding_rupture, prying_action, flexural_14th, flexural_15th
from numpy import atan,radians


ao1_angle_bearing_args = {
    "F_u": 58,
    "d_bolt": 7/8,
    "t": 0.375,
    "P_u": 5,
    "V_u": 30,
    "S_r": 3,
    "N_r": 4,
    "S_c": 0,
    "N_c": 1,
    "L_ev": 1.25,
    "L_eh": 1.25,
    "d_v": 0.938,
    "d_h": 0.938,
    "phi": 0.75,
    "c": 4
}
ao1_beam_bearing_args = {
    "F_u": 65,
    "d_bolt": 7/8,
    "t": 0.25,
    "P_u": 5,
    "V_u": 30,
    "S_r": 3,
    "N_r": 4,
    "S_c": 0,
    "N_c": 1,
    "L_ev": 1.25,
    "L_eh": 1.75,
    "d_v": 0.938,
    "d_h": 0.938,
    "phi": 0.75
}
ao1_0sl_angle_bearing_args = {
    "F_u": 58,
    "d_bolt": 7/8,
    "t": 0.375,
    "P_u": 0,
    "V_u": 30,
    "S_r": 3,
    "N_r": 4,
    "S_c": 0,
    "N_c": 1,
    "L_ev": 1.25,
    "L_eh": 1.25,
    "d_v": 0.938,
    "d_h": 0.938,
    "phi": 0.75,
    'c':4
}

a02_osl_bearing = {
    "F_u": 58,
    "d_bolt": 7/8,
    "t": 0.5,
    "P_u": 0,
    "V_u": 70,
    "S_r": 3,
    "N_r": 4,
    "S_c": 0,
    "N_c": 1,
    "L_ev": 1.25,
    "L_eh": 1.428,
    "d_v": 0.938,
    "d_h": 0.938,
    "phi": 0.75,
    'c':4
}

a03_nsl_plate_bearing = {
    "F_u": 65,
    "d_bolt": 1+1/8,
    "t": 0.625,
    "P_u": 60,
    "V_u": 90,
    "S_r": 3,
    "N_r": 5,
    "S_c": 3,
    "N_c": 3,
    "L_ev": 1.5,
    "L_eh": 2,
    "d_v":  1.25,
    "d_h": 1.5,
    "phi": 0.75,
    'c':5.226
}

ao3_block_shear_plate = {
    "P_u": 60, 
    "V_u": 90, 
    "F_y": 50, 
    "F_u": 65, 
    "t": 0.625, 
    "N_r": 5, 
    "S_r": 3, 
    "N_c": 3, 
    "S_c": 3, 
    "L_ev": 1.5, 
    "L_eh": 2, 
    "d_hole": 1.5 + 1/16, 
    "phi": 0.75,
    "coped":True,
    "n_members": 2
}
ao1_nsl_block_shear_plate = {
    "P_u": 5, 
    "V_u": 30, 
    "F_y": 36, 
    "F_u": 58, 
    "t": 0.375, 
    "N_r": 4, 
    "S_r": 3, 
    "N_c": 1, 
    "S_c": 3, 
    "L_ev": 1.25, 
    "L_eh": 1.25, 
    "d_v": 0.938, 
    "d_h": 0.938,
    "phi": 0.75,
    "coped":False,
    "n_members": 2
}
# block_shear(**ao1_nsl_block_shear_plate)

ao1_shear_yielding_rupture_plate_nsl = {
    "N_r": 4,
    "S_r": 3,
    "L_ev": 1.25,
    "t": 0.375,
    "d_b": 7/8,
    "F_y": 36,
    "F_u": 58,
    "phi": 0.75,
    "n_members": 2,
    "coped": 2
}
ao1_tensile_yielding_rupture_plate_nsl = {
    "N_r": 4,
    "S_r": 3,
    "L_ev": 1.25,
    "t": 0.375,
    "d_b": 7/8,
    "F_y": 36,
    "F_u": 58,
    "phi": 0.75,
    "n_members": 2,
}

# flexural(l = 3.5,k_a = 3/4 ,L_eh = 1.25, N_c = 1, _S_c = 3, t = 0.375, N_r = 4, S_r = 3, d_b = 7/8, d = None, F_y = 36, F_u = 58, e_override=None)
# a01_prying_osl_angle = prying_action(F_nt = 90,F_nv = 54, n_bolts = 4,t = 0.375, F_u = 58, N_t = 4, L = 3, L_eh = 1.25, L_ev = 1.25, d_bolt = 7/8, d_v = 0.938, S_r = 3, ga = 3.75, P_u = 5,V_u = 30, prying= True)
# a02_prying_osl_support = prying_action(F_nt = 90,F_nv = 54, n_bolts = 4,t = 0.355, F_u = 65, N_t = 4, L = 0, L_eh = 1.25, L_ev = 3, d_bolt = 7/8, d_v = 0.938, S_r = 3, ga = 3.75, P_u = 5,V_u = 30, prying= True,bf = 7.5)

a03_1_plate_flexure = {
    "member_type": "PL",
    "a": 10,
    "t": 0.625,
    "F_y": 50,
    "E": 29000,
    "F_u": 65,
    "N_r":5,
    "S_r":3,
    "d_bolt":1.25,
    "L_ev":1.5,
    "F_u":65,
    "d_b": 18

}

a03_flexural_nsl = flexural_15th(**a03_1_plate_flexure)

In [11]:
from steel_lib.generator_combination import generate_bolt_configurations, get_mapping_info

# Inspect integer-to-string mappings for clarity
mappings = get_mapping_info()
mappings

# Mock input space for bolt configuration generation
mock_inputs = {
    "bolt_size": np.array([16, 20, 24], dtype=np.int64),
    "bolt_grade_id": np.array([0, 1, 2], dtype=np.int64),  # 0=a325_n, 1=a325_x, 2=a490_n
    "member_a_BHT_id": np.array([0, 1], dtype=np.int64),    # 0=STD, 1=OVS
    "member_b_BHT_id": np.array([0, 2], dtype=np.int64),    # 0=STD, 2=SSL
    "N_r": np.array([1, 2], dtype=np.int64),
    "S_r": np.array([60.0, 80.0], dtype=np.float64),
    "N_c": np.array([2, 3], dtype=np.int64),
    "S_c": np.array([60.0, 90.0], dtype=np.float64),
    "L_ev": np.array([30.0, 45.0], dtype=np.float64),
    "L_eh": np.array([30.0, 40.0], dtype=np.float64),
    "Ga": np.array([0.0, 5.0], dtype=np.float64)
}

# Generate the full combination table using the mock inputs
bolt_configs = generate_bolt_configurations(**mock_inputs)

# Peek at a handful of generated combinations


In [12]:
import time

tests_run = []
skipped_tests = {}
pending_tests = {}

if "np" not in globals():
    import numpy as np
if "create_aisc_section_selector" not in globals():
    from steel_lib.section_properties import create_aisc_section_selector
if "generate_bolt_configurations" not in globals() or "get_mapping_info" not in globals():
    from steel_lib.generator_combination import generate_bolt_configurations, get_mapping_info
if "mock_inputs" not in globals():
    mock_inputs = {
        "bolt_size": np.array([16, 20, 24], dtype=np.int64),
        "bolt_grade_id": np.array([0, 1, 2], dtype=np.int64),
        "member_a_BHT_id": np.array([0, 1], dtype=np.int64),
        "member_b_BHT_id": np.array([0, 2], dtype=np.int64),
        "N_r": np.array([1, 2], dtype=np.int64),
        "S_r": np.array([60.0, 80.0], dtype=np.float64),
        "N_c": np.array([2, 3], dtype=np.int64),
        "S_c": np.array([60.0, 90.0], dtype=np.float64),
        "L_ev": np.array([30.0, 45.0], dtype=np.float64),
        "L_eh": np.array([30.0, 40.0], dtype=np.float64),
        "Ga": np.array([0.0, 5.0], dtype=np.float64)
    }
    print("Mock input space recreated locally (previous cell not run).")

# --- Test 1: AISC selector build/regression ---
print("[TEST] Section selector build and sample query")
selector_ref = None
selector_elapsed_ms = None

if "selector" in globals() and isinstance(selector, dict) and "database_info" in selector:
    selector_ref = selector
    print("  Reusing selector instance already in memory.")
else:
    selector_start = time.perf_counter()
    selector_ref = create_aisc_section_selector("aisc-shapes-database-v16.0.xlsx")
    selector_elapsed_ms = (time.perf_counter() - selector_start) * 1e3
    print(f"  Load time: {selector_elapsed_ms:.1f} ms")
    selector = selector_ref
tests_run.append("section_selector")

aisc_sample = selector_ref["select_by_properties"]({"W": {"min": 8.0, "max": 10.0}})
match_count = aisc_sample.get("count", len(aisc_sample.get("designations", [])))
print(f"  Matches: {match_count}")
if match_count:
    print("  Sample designations:", aisc_sample["designations"][:5])
else:
    print("  Selector returned no shapes for the test filter.")

# --- Test 2: Bolt configuration generator ---
print("[TEST] Bolt configuration generator")
bolt_start = time.perf_counter()
bolt_configs = generate_bolt_configurations(**mock_inputs)
bolt_elapsed_ms = (time.perf_counter() - bolt_start) * 1e3
print(f"  Combinations: {len(bolt_configs['bolt_size'])} in {bolt_elapsed_ms:.1f} ms")
preview_keys = ("bolt_size", "bolt_grade", "F_nv", "F_nt")
print({key: bolt_configs[key][:2] for key in preview_keys if key in bolt_configs})
tests_run.append("bolt_configuration_generator")

synthetic_length = len(bolt_configs["bolt_size"]) if isinstance(bolt_configs, dict) and "bolt_size" in bolt_configs else 0
if synthetic_length == 0:
    fallback_length = match_count if match_count else 1
    synthetic_length = fallback_length
bolt_diam_base = np.asarray(bolt_configs["bolt_size"], dtype=np.float64) if isinstance(bolt_configs, dict) and "bolt_size" in bolt_configs else np.full(synthetic_length, 20.0, dtype=np.float64)
if bolt_diam_base.shape[0] < synthetic_length:
    bolt_diam = np.resize(bolt_diam_base, synthetic_length)
else:
    bolt_diam = bolt_diam_base
member_code_pattern = np.tile(np.array([0, 1], dtype=np.int32), synthetic_length // 2 + 1)[:synthetic_length]
coped_pattern = np.tile(np.array([True, False]), synthetic_length // 2 + 1)[:synthetic_length]
coped_syr_pattern = np.tile(np.array([0, 1, 2], dtype=np.int64), synthetic_length // 3 + 1)[:synthetic_length]
prying_pattern = np.tile(np.array([False, True]), synthetic_length // 2 + 1)[:synthetic_length]
synthetic_inputs = {
    "F_u": np.full(synthetic_length, 58.0, dtype=np.float64),
    "P_u": np.full(synthetic_length, 50.0, dtype=np.float64),
    "V_u": np.full(synthetic_length, 30.0, dtype=np.float64),
    "dv": bolt_diam * 1.1,
    "dh": bolt_diam * 1.1,
    "c": np.full(synthetic_length, 4.0, dtype=np.float64),
    "N_shear_planes": np.full(synthetic_length, 2, dtype=np.int64),
    "F_y": np.full(synthetic_length, 36.0, dtype=np.float64),
    "coped": coped_pattern.astype(bool),
    "d_b": bolt_diam * 1.12,
    "n_members": np.full(synthetic_length, 2, dtype=np.int64),
    "coped_syr": coped_syr_pattern,
    "l": np.full(synthetic_length, 3.5, dtype=np.float64),
    "k_a": np.full(synthetic_length, 0.75, dtype=np.float64),
    "e_override": np.full(synthetic_length, 0.3, dtype=np.float64),
    "F_nv_prying": np.full(synthetic_length, 54.0, dtype=np.float64),
    "N_t": np.full(synthetic_length, 4, dtype=np.int64),
    "L": np.full(synthetic_length, 75.0, dtype=np.float64),
    "prying": prying_pattern.astype(bool),
    "member_type_codes": member_code_pattern.astype(np.int32),
    "a": np.full(synthetic_length, 10.0, dtype=np.float64),
    "E": np.full(synthetic_length, 29000.0, dtype=np.float64)
}

# --- Test 3: Limit state batch wrapper ---
print("[TEST] Limit state batch (vectorised wrapper)")

required_inputs = [
    "F_u", "d_bolt", "t", "P_u", "V_u", "S_r", "N_r", "S_c", "N_c", "L_ev", "L_eh",
    "dv", "dh", "c", "F_nv", "A_bolt", "N_shear_planes", "F_y", "coped", "d_b",
    "n_members", "coped_syr", "l", "k_a", "d", "e_override", "F_nt", "F_nv_prying",
    "N_t", "L", "ga", "prying", "member_type_codes", "a", "E"
 ]
input_mapping = {
    "d_bolt": ("bolt_configs", "bolt_size"),
    "S_r": ("bolt_configs", "S_r"),
    "N_r": ("bolt_configs", "N_r"),
    "S_c": ("bolt_configs", "S_c"),
    "N_c": ("bolt_configs", "N_c"),
    "L_ev": ("bolt_configs", "L_ev"),
    "L_eh": ("bolt_configs", "L_eh"),
    "F_nv": ("bolt_configs", "F_nv"),
    "F_nt": ("bolt_configs", "F_nt"),
    "ga": ("bolt_configs", "Ga"),
    "d": ("aisc_sample", "d"),
    "t": ("aisc_sample", "t"),
    "F_u": ("synthetic", "F_u"),
    "P_u": ("synthetic", "P_u"),
    "V_u": ("synthetic", "V_u"),
    "dv": ("synthetic", "dv"),
    "dh": ("synthetic", "dh"),
    "c": ("synthetic", "c"),
    "N_shear_planes": ("synthetic", "N_shear_planes"),
    "F_y": ("synthetic", "F_y"),
    "coped": ("synthetic", "coped"),
    "d_b": ("synthetic", "d_b"),
    "n_members": ("synthetic", "n_members"),
    "coped_syr": ("synthetic", "coped_syr"),
    "l": ("synthetic", "l"),
    "k_a": ("synthetic", "k_a"),
    "e_override": ("synthetic", "e_override"),
    "F_nv_prying": ("synthetic", "F_nv_prying"),
    "N_t": ("synthetic", "N_t"),
    "L": ("synthetic", "L"),
    "prying": ("synthetic", "prying"),
    "member_type_codes": ("synthetic", "member_type_codes"),
    "a": ("synthetic", "a"),
    "E": ("synthetic", "E"),
    "A_bolt": ("bolt_configs", "bolt_size")  # placeholder to allow derived mapping later
}
data_sources = {
    "bolt_configs": bolt_configs,
    "aisc_sample": aisc_sample,
    "synthetic": synthetic_inputs
}

computed_inputs = {}
missing_inputs = []

for name in required_inputs:
    mapping = input_mapping.get(name)
    if mapping:
        src_name, key = mapping
        source_dict = data_sources.get(src_name, {})
        values = source_dict.get(key) if isinstance(source_dict, dict) else None
        if values is None or (hasattr(values, "__len__") and len(values) == 0):
            missing_inputs.append(name)
        else:
            computed_inputs[name] = np.asarray(values)
    else:
        missing_inputs.append(name)

if missing_inputs:
    print("[ACTION REQUIRED] Missing inputs for limit state batch test:")
    for item in missing_inputs:
        print(f"  - {item}")
    print("  Available section fields:", sorted(key for key in aisc_sample.keys() if key not in ("count",)))
    print("  Available bolt fields:", sorted(bolt_configs.keys()))
    print("Please advise how to derive or supply these values using only the selector and generator outputs.")
    pending_tests["limit_state_batch"] = missing_inputs
else:
    # Align sample size from available inputs
    candidate_lengths = [len(arr) for arr in computed_inputs.values() if hasattr(arr, "__len__")]
    num_sample = min(candidate_lengths) if candidate_lengths else 0
    if num_sample == 0:
        print("[SKIP] Limit state batch: derived inputs had zero length after alignment")
        skipped_tests["limit_state_batch"] = ["empty_inputs"]
    else:
        # Trim arrays to uniform length
        for key, arr in computed_inputs.items():
            if hasattr(arr, "__len__") and len(arr) > num_sample:
                computed_inputs[key] = np.asarray(arr)[:num_sample]
            else:
                computed_inputs[key] = np.asarray(arr)
        if "ga" in computed_inputs:
            computed_inputs["ga"] = np.where(computed_inputs["ga"] <= 0.0, 3.0, computed_inputs["ga"])
        if "t" in computed_inputs:
            computed_inputs["t"] = np.where(computed_inputs["t"] <= 0.0, 0.375, computed_inputs["t"])
        if "d" in computed_inputs:
            computed_inputs["d"] = np.where(computed_inputs["d"] <= 0.0, 8.0, computed_inputs["d"])
        # Derived fields
        computed_inputs["A_bolt"] = np.pi * (computed_inputs["d_bolt"] / 2.0)**2
        computed_inputs["n_bolts"] = computed_inputs["N_r"] * computed_inputs["N_c"]
        # Output buffers
        results_bearing = np.zeros(num_sample, dtype=np.float64)
        results_shear = np.zeros(num_sample, dtype=np.float64)
        results_block = np.zeros(num_sample, dtype=np.float64)
        results_syr = np.zeros(num_sample, dtype=np.float64)
        results_flexural = np.zeros(num_sample, dtype=np.float64)
        results_prying = np.zeros(num_sample, dtype=np.float64)
        results_flexural_15th = np.zeros(num_sample, dtype=np.float64)
        phi = np.ones(num_sample, dtype=np.float64) * 0.75
        limit_start = time.perf_counter()
        test_bolt_strength_function(
            computed_inputs["F_u"],
            computed_inputs["d_bolt"],
            computed_inputs["t"],
            computed_inputs["P_u"],
            computed_inputs["V_u"],
            computed_inputs["S_r"],
            computed_inputs["N_r"],
            computed_inputs["S_c"],
            computed_inputs["N_c"],
            computed_inputs["L_ev"],
            computed_inputs["L_eh"],
            computed_inputs["dv"],
            computed_inputs["dh"],
            computed_inputs["c"],
            computed_inputs["F_nv"],
            computed_inputs["A_bolt"],
            computed_inputs["N_shear_planes"],
            computed_inputs["F_y"],
            computed_inputs["coped"],
            computed_inputs["d_b"],
            computed_inputs["n_members"],
            computed_inputs["coped_syr"],
            computed_inputs["l"],
            computed_inputs["k_a"],
            computed_inputs["d"],
            computed_inputs["e_override"],
            computed_inputs["F_nt"],
            computed_inputs["F_nv_prying"],
            computed_inputs["n_bolts"],
            computed_inputs["N_t"],
            computed_inputs["L"],
            computed_inputs["ga"],
            computed_inputs["prying"],
            computed_inputs["member_type_codes"],
            computed_inputs["a"],
            computed_inputs["E"],
            phi,
            results_bearing,
            results_shear,
            results_block,
            results_syr,
            results_flexural,
            results_prying,
            results_flexural_15th,
            num_sample
)
        limit_elapsed_ms = (time.perf_counter() - limit_start) * 1e3
        print(f"  Runtime: {limit_elapsed_ms:.1f} ms for {num_sample} cases")
        tests_run.append("limit_state_batch")

# --- Summary ---
print("\nSummary")
for name in tests_run:
    print(f"  PASS: {name}")
for name, missing in skipped_tests.items():
    print(f"  SKIP: {name} (missing {missing})")
for name, missing in pending_tests.items():
    print(f"  PENDING: {name} (needs inputs: {missing})")

[TEST] Section selector build and sample query
Loading AISC Shapes Database...
Loaded 2299 sections from AISC database
  Load time: 3525.0 ms
  Matches: 119
  Sample designations: ['W8X10' 'W6X9' 'W6X8.5' 'M12X10' 'M10X9']
[TEST] Bolt configuration generator
  Combinations: 4608 in 24.7 ms
{'bolt_size': array([16., 16.]), 'bolt_grade': array(['a325_n', 'a325_n'], dtype='<U6'), 'F_nv': array([3.72316894e+08, 3.72316894e+08]), 'F_nt': array([6.20528156e+08, 6.20528156e+08])}
[TEST] Limit state batch (vectorised wrapper)
Loaded 2299 sections from AISC database
  Load time: 3525.0 ms
  Matches: 119
  Sample designations: ['W8X10' 'W6X9' 'W6X8.5' 'M12X10' 'M10X9']
[TEST] Bolt configuration generator
  Combinations: 4608 in 24.7 ms
{'bolt_size': array([16., 16.]), 'bolt_grade': array(['a325_n', 'a325_n'], dtype='<U6'), 'F_nv': array([3.72316894e+08, 3.72316894e+08]), 'F_nt': array([6.20528156e+08, 6.20528156e+08])}
[TEST] Limit state batch (vectorised wrapper)


TypeError: too many arguments: expected 42, got 45

In [13]:
import numpy as np
from steel_lib.generator_combination import generate_bolt_configurations, get_mapping_info
from steel_lib.aisc_14th import bolt_bearing, block_shear, shear_yielding_rupture, prying_action, flexural_14th, flexural_15th
aisc = create_aisc_section_selector('aisc-shapes-database-v16.0.xlsx')
sections = aisc['select_by_properties']({
    "designation": ["W14X53"],
})
mock_inputs = {
    "bolt_size": np.array([16, 20, 24], dtype=np.int64),
    "bolt_grade_id": np.array([0, 1, 2], dtype=np.int64),
    "member_a_BHT_id": np.array([0, 1], dtype=np.int64),
    "member_b_BHT_id": np.array([0, 2], dtype=np.int64),
    "N_r": np.array([1, 2], dtype=np.int64),
    "S_r": np.array([60.0, 80.0], dtype=np.float64),
    "N_c": np.array([2, 3], dtype=np.int64),
    "S_c": np.array([60.0, 90.0], dtype=np.float64),
    "L_ev": np.array([30.0, 45.0], dtype=np.float64),
    "L_eh": np.array([30.0, 40.0], dtype=np.float64),
    "Ga": np.array([0.0, 5.0], dtype=np.float64)
}
generate_bolt_configurations(**mock_inputs)
shear_yielding_rupture

Loading AISC Shapes Database...
Loaded 2299 sections from AISC database
Loaded 2299 sections from AISC database


CPUDispatcher(<function shear_yielding_rupture at 0x0000014124922A20>)

# Plate Member Generator System

This demonstrates the new plate configuration generator that works similarly to the bolt configuration system.

In [1]:
# Import the new plate generator system
from steel_lib.plate_generator import (
    generate_plate_configurations,
    generate_shear_plates,
    generate_flange_plates,
    generate_stiffener_plates,
    generate_gusset_plates,
    get_plate_mapping_info,
    STANDARD_PLATE_THICKNESSES
)

# Get mapping information
plate_mappings = get_plate_mapping_info()
print("Plate System Configuration")
print("=" * 60)
print(f"Available Plate Types: {plate_mappings['plate_types']}")
print(f"Available Steel Grades: {plate_mappings['plate_grades']}")
print(f"\nStandard Thicknesses (inches):")
print(STANDARD_PLATE_THICKNESSES)
print()

# Material properties
print("\nMaterial Properties:")
for grade, props in plate_mappings['material_properties'].items():
    print(f"  {grade:10s}: F_y = {props['F_y']:3.0f} ksi, F_u = {props['F_u']:3.0f} ksi")


Plate System Configuration
Available Plate Types: {0: 'SHEAR', 1: 'FLANGE', 2: 'STIFFENER', 3: 'GUSSET', 4: 'BASE', 5: 'DOUBLER'}
Available Steel Grades: {0: 'A36', 1: 'A572_50', 2: 'A992', 3: 'A588', 4: 'A514'}

Standard Thicknesses (inches):
[0.1875 0.25   0.3125 0.375  0.4375 0.5    0.5625 0.625  0.6875 0.75
 0.875  1.     1.125  1.25   1.375  1.5    1.75   2.     2.25   2.5
 3.     4.    ]


Material Properties:
  A36       : F_y =  36 ksi, F_u =  58 ksi
  A572_50   : F_y =  50 ksi, F_u =  65 ksi
  A992      : F_y =  50 ksi, F_u =  65 ksi
  A588      : F_y =  50 ksi, F_u =  70 ksi
  A514      : F_y = 100 ksi, F_u = 110 ksi


In [2]:
# Example 1: Generate Shear Plate Configurations
# Similar to bolt configurations - define parameter ranges and generate all combinations

print("EXAMPLE 1: Shear Plates")
print("=" * 60)

shear_plates = generate_shear_plates(
    plate_grade_id=[0, 1],              # A36 and A572_50
    thickness=[0.25, 0.375, 0.5],       # 1/4", 3/8", 1/2"
    width=[3.5, 5.0, 6.0],              # 3.5", 5", 6"
    length=[12.0, 18.0, 24.0]           # 12", 18", 24"
)

print(f"Generated {len(shear_plates['thickness'])} shear plate combinations")
print(f"\nAvailable properties: {list(shear_plates.keys())}")

# Show first 5 configurations
print(f"\nFirst 5 Shear Plate Configurations:")
print("-" * 60)
for i in range(min(5, len(shear_plates['thickness']))):
    print(f"Config {i+1}:")
    print(f"  Material: {shear_plates['plate_grade'][i]}")
    print(f"  Size: PL {shear_plates['width'][i]:.2f}\" x {shear_plates['thickness'][i]:.3f}\" x {shear_plates['length'][i]:.1f}\"")
    print(f"  F_y = {shear_plates['F_y'][i]:.0f} ksi, F_u = {shear_plates['F_u'][i]:.0f} ksi")
    print(f"  Area = {shear_plates['area'][i]:.3f} in², Weight = {shear_plates['total_weight'][i]:.2f} lb")
    print(f"  φV_n (gross) = {shear_plates['phi_V_n_gross'][i]:.1f} kips")
    print()


EXAMPLE 1: Shear Plates
Generated 54 shear plate combinations

Available properties: ['plate_type_id', 'plate_grade_id', 'thickness', 'width', 'length', 'plate_type', 'plate_grade', 'F_y', 'F_u', 'E', 'area', 'weight_plf', 'total_weight', 'S_x', 'I_x', 'V_n_gross', 'phi_V_n_gross']

First 5 Shear Plate Configurations:
------------------------------------------------------------
Config 1:
  Material: A36
  Size: PL 3.50" x 0.250" x 12.0"
  F_y = 36 ksi, F_u = 58 ksi
  Area = 0.875 in², Weight = 2.98 lb
  φV_n (gross) = 18.9 kips

Config 2:
  Material: A36
  Size: PL 3.50" x 0.250" x 18.0"
  F_y = 36 ksi, F_u = 58 ksi
  Area = 0.875 in², Weight = 4.47 lb
  φV_n (gross) = 18.9 kips

Config 3:
  Material: A36
  Size: PL 3.50" x 0.250" x 24.0"
  F_y = 36 ksi, F_u = 58 ksi
  Area = 0.875 in², Weight = 5.95 lb
  φV_n (gross) = 18.9 kips

Config 4:
  Material: A36
  Size: PL 5.00" x 0.250" x 12.0"
  F_y = 36 ksi, F_u = 58 ksi
  Area = 1.250 in², Weight = 4.25 lb
  φV_n (gross) = 27.0 kips

Con

In [3]:
# Example 2: Generate Flange Plate Configurations
# For moment connections - typically thicker plates

print("EXAMPLE 2: Flange Plates (Moment Connections)")
print("=" * 60)

flange_plates = generate_flange_plates(
    plate_grade_id=[1, 2],              # A572_50 and A992
    thickness=[0.75, 1.0, 1.25],        # 3/4", 1", 1-1/4"
    width=[8.0, 10.0, 12.0],            # 8", 10", 12"
    length=[15.0, 18.0, 24.0]           # 15", 18", 24"
)

print(f"Generated {len(flange_plates['thickness'])} flange plate combinations")

# Show first 3 configurations with capacity information
print(f"\nFirst 3 Flange Plate Configurations:")
print("-" * 60)
for i in range(min(3, len(flange_plates['thickness']))):
    print(f"Config {i+1}:")
    print(f"  Material: {flange_plates['plate_grade'][i]}")
    print(f"  Size: PL {flange_plates['width'][i]:.2f}\" x {flange_plates['thickness'][i]:.3f}\" x {flange_plates['length'][i]:.1f}\"")
    print(f"  φP_n (yield) = {flange_plates['phi_P_n_yield'][i]:.1f} kips")
    print(f"  φP_n (rupture) = {flange_plates['phi_P_n_rupture'][i]:.1f} kips")
    print(f"  Z_x = {flange_plates['Z_x'][i]:.2f} in³")
    print()


EXAMPLE 2: Flange Plates (Moment Connections)
Generated 54 flange plate combinations

First 3 Flange Plate Configurations:
------------------------------------------------------------
Config 1:
  Material: A572_50
  Size: PL 8.00" x 0.750" x 15.0"
  φP_n (yield) = 270.0 kips
  φP_n (rupture) = 292.5 kips
  Z_x = 1.12 in³

Config 2:
  Material: A572_50
  Size: PL 8.00" x 0.750" x 18.0"
  φP_n (yield) = 270.0 kips
  φP_n (rupture) = 292.5 kips
  Z_x = 1.12 in³

Config 3:
  Material: A572_50
  Size: PL 8.00" x 0.750" x 24.0"
  φP_n (yield) = 270.0 kips
  φP_n (rupture) = 292.5 kips
  Z_x = 1.12 in³



In [4]:
# Example 3: Generate Stiffener Plate Configurations
# For web or column stiffeners

print("EXAMPLE 3: Stiffener Plates")
print("=" * 60)

stiffener_plates = generate_stiffener_plates(
    plate_grade_id=[0, 1],              # A36 and A572_50
    thickness=[0.375, 0.5, 0.625],      # 3/8", 1/2", 5/8"
    width=[4.0, 5.0, 6.0],              # 4", 5", 6"
    height=[12.0, 18.0, 24.0]           # 12", 18", 24"
)

print(f"Generated {len(stiffener_plates['thickness'])} stiffener combinations")

# Show first 3 configurations with design checks
print(f"\nFirst 3 Stiffener Configurations:")
print("-" * 60)
for i in range(min(3, len(stiffener_plates['thickness']))):
    print(f"Config {i+1}:")
    print(f"  Material: {stiffener_plates['plate_grade'][i]}")
    print(f"  Size: PL {stiffener_plates['width'][i]:.2f}\" x {stiffener_plates['thickness'][i]:.3f}\" x {stiffener_plates['height'][i]:.1f}\"")
    print(f"  φR_n (bearing, 2 stiffeners) = {stiffener_plates['phi_R_n_bearing'][i]:.1f} kips")
    print(f"  b/t ratio = {stiffener_plates['b_t_ratio'][i]:.2f} (limit = {stiffener_plates['b_t_limit'][i]:.2f})")
    print(f"  KL/r = {stiffener_plates['kl_r'][i]:.1f}")
    print()


EXAMPLE 3: Stiffener Plates
Generated 54 stiffener combinations

First 3 Stiffener Configurations:
------------------------------------------------------------
Config 1:
  Material: A36
  Size: PL 4.00" x 0.375" x 12.0"
  φR_n (bearing, 2 stiffeners) = 145.8 kips
  b/t ratio = 5.33 (limit = 15.89)
  KL/r = 83.1

Config 2:
  Material: A36
  Size: PL 4.00" x 0.375" x 18.0"
  φR_n (bearing, 2 stiffeners) = 145.8 kips
  b/t ratio = 5.33 (limit = 15.89)
  KL/r = 124.7

Config 3:
  Material: A36
  Size: PL 4.00" x 0.375" x 24.0"
  φR_n (bearing, 2 stiffeners) = 145.8 kips
  b/t ratio = 5.33 (limit = 15.89)
  KL/r = 166.3



In [ ]:
# Example 4: Generate Gusset Plate Configurations
# For bracing connections

print("EXAMPLE 4: Gusset Plates")
print("=" * 60)

gusset_plates = generate_gusset_plates(
    plate_grade_id=[0, 1],              # A36 and A572_50
    thickness=[0.5, 0.625, 0.75],       # 1/2", 5/8", 3/4"
    width=[18.0, 24.0, 30.0],           # 18", 24", 30"
    length=[18.0, 24.0, 36.0],          # 18", 24", 36"
    angle=[35.0, 45.0, 55.0]            # Brace angles in degrees
)

print(f"Generated {len(gusset_plates['thickness'])} gusset plate combinations")

# Show first 3 configurations
print(f"\nFirst 3 Gusset Plate Configurations:")
print("-" * 60)
for i in range(min(3, len(gusset_plates['thickness']))):
    print(f"Config {i+1}:")
    print(f"  Material: {gusset_plates['plate_grade'][i]}")
    print(f"  Size: PL {gusset_plates['width'][i]:.2f}\" x {gusset_plates['thickness'][i]:.3f}\" x {gusset_plates['length'][i]:.1f}\"")
    print(f"  Brace Angle: {gusset_plates['angle'][i]:.1f}°")
    print(f"  Weight = {gusset_plates['total_weight'][i]:.2f} lb")
    print(f"  Slenderness = {gusset_plates['slenderness'][i]:.1f}")
    print()


EXAMPLE 4: Gusset Plates


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x00000202B69F2990>>
Traceback (most recent call last):
  File "c:\Users\peter\OneDrive\Documents\asdas\venv\Lib\site-packages\ipykernel\ipkernel.py", line 796, in _clean_thread_parent_frames
    active_threads = {thread.ident for thread in threading.enumerate()}
                                                 ^^^^^^^^^^^^^^^^^^^^^
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.11_3.11.2544.0_x64__qbz5n2kfra8p0\Lib\threading.py", line 1501, in enumerate
    def enumerate():
    
KeyboardInterrupt: 


In [5]:
# Example 5: Combined Usage - Integrate with Bolt Configurations
# Similar to how you combine sections and bolts, you can combine plates and bolts

print("EXAMPLE 5: Combined Plate + Bolt System")
print("=" * 60)

# Generate a simple set of shear plates
plates = generate_shear_plates(
    plate_grade_id=[0],                 # A36 only
    thickness=[0.375, 0.5],             # 3/8" and 1/2"
    width=[5.0],                        # 5" wide
    length=[18.0]                       # 18" long
)

# Generate corresponding bolt configurations
from steel_lib.generator_combination import generate_bolt_configurations

bolts = generate_bolt_configurations(
    bolt_size=np.array([0.875], dtype=np.float64),      # 7/8" bolts
    bolt_grade_id=np.array([0], dtype=np.int64),        # A325-N
    member_a_BHT_id=np.array([0], dtype=np.int64),      # STD holes
    member_b_BHT_id=np.array([0], dtype=np.int64),      # STD holes
    N_r=np.array([4], dtype=np.int64),                  # 4 rows
    S_r=np.array([3.0], dtype=np.float64),              # 3" spacing
    N_c=np.array([1], dtype=np.int64),                  # 1 column
    S_c=np.array([3.0], dtype=np.float64),              # N/A
    L_ev=np.array([1.25], dtype=np.float64),            # Edge distances
    L_eh=np.array([1.25], dtype=np.float64),
    Ga=np.array([0.0], dtype=np.float64)
)

print(f"Plates: {len(plates['thickness'])} configurations")
print(f"Bolts: {len(bolts['bolt_size'])} configurations")
print()

# Show combined configuration
print("Combined Configuration:")
print("-" * 60)
for i in range(len(plates['thickness'])):
    print(f"\nPlate Config {i+1}:")
    print(f"  Plate: {plates['plate_grade'][i]} PL {plates['width'][i]:.2f}\"x{plates['thickness'][i]:.3f}\"x{plates['length'][i]:.1f}\"")
    print(f"  Bolt: ({bolts['N_r'][0]}x{bolts['N_c'][0]}) {bolts['bolt_size'][0]:.3f}\" φ {bolts['bolt_grade'][0]}")
    print(f"  Plate φV_n = {plates['phi_V_n_gross'][i]:.1f} kips")
    print(f"  Bolt F_nv = {bolts['F_nv'][0]:.1f} ksi")
    
    # Check if plate can accommodate bolt pattern
    min_plate_length = 2 * bolts['L_ev'][0] + (bolts['N_r'][0] - 1) * bolts['S_r'][0]
    print(f"  Min plate length needed: {min_plate_length:.2f}\" (actual: {plates['length'][i]:.1f}\") ✓" if plates['length'][i] >= min_plate_length else f"  ALERT: Plate too short!")


EXAMPLE 5: Combined Plate + Bolt System


NameError: name 'np' is not defined

In [6]:
# Example 6: Using with AISC Sections and Your Limit State Functions
# Full integration example showing plates + sections + bolts

print("EXAMPLE 6: Complete Connection Design Space")
print("=" * 60)

# Select some beam sections
beam_sections = aisc['select_by_properties']({
    "designation": ["W16X26", "W18X35"],
})

# Generate connection plates
connection_plates = generate_shear_plates(
    plate_grade_id=[1],                 # A572_50
    thickness=[0.375, 0.5],             # 3/8", 1/2"
    width=[5.0, 5.5],                   # 5", 5.5"
    length=[15.0, 18.0]                 # 15", 18"
)

# Generate bolt patterns
bolt_patterns = generate_bolt_configurations(
    bolt_size=np.array([0.75, 0.875], dtype=np.float64),
    bolt_grade_id=np.array([0], dtype=np.int64),
    member_a_BHT_id=np.array([0], dtype=np.int64),
    member_b_BHT_id=np.array([0], dtype=np.int64),
    N_r=np.array([3, 4], dtype=np.int64),
    S_r=np.array([3.0], dtype=np.float64),
    N_c=np.array([1], dtype=np.int64),
    S_c=np.array([3.0], dtype=np.float64),
    L_ev=np.array([1.25], dtype=np.float64),
    L_eh=np.array([1.5], dtype=np.float64),
    Ga=np.array([0.0], dtype=np.float64)
)

print(f"Design Space:")
print(f"  Beam Sections: {beam_sections['count']}")
print(f"  Connection Plates: {len(connection_plates['thickness'])}")
print(f"  Bolt Patterns: {len(bolt_patterns['bolt_size'])}")
print(f"  Total Combinations: {beam_sections['count'] * len(connection_plates['thickness']) * len(bolt_patterns['bolt_size'])}")
print()

# Show sample combination
print("Sample Connection Configuration:")
print("-" * 60)
print(f"Beam: {beam_sections['designations'][0]}")
print(f"  d = {beam_sections['d'][0]:.2f}\", tw = {beam_sections['tw'][0]:.3f}\"")
print(f"\nConnection Plate: {connection_plates['plate_grade'][0]}")
print(f"  PL {connection_plates['width'][0]:.2f}\"x{connection_plates['thickness'][0]:.3f}\"x{connection_plates['length'][0]:.1f}\"")
print(f"  F_y = {connection_plates['F_y'][0]:.0f} ksi, φV_n = {connection_plates['phi_V_n_gross'][0]:.1f} kips")
print(f"\nBolts: {bolt_patterns['N_r'][0]}x{bolt_patterns['N_c'][0]} @ {bolt_patterns['S_r'][0]:.1f}\" o.c.")
print(f"  Size: {bolt_patterns['bolt_size'][0]:.3f}\" φ {bolt_patterns['bolt_grade'][0]}")
print(f"  F_nv = {bolt_patterns['F_nv'][0]:.1f} ksi")

# This design space can now be fed into your limit state functions
# for comprehensive capacity checks (bearing, block shear, etc.)


EXAMPLE 6: Complete Connection Design Space


NameError: name 'aisc' is not defined

# Load Path System - Track Forces Through Connections

The load path system models how loads transfer from one member to another through connection elements (bolts, plates, welds).

In [9]:
# Example 1: Basic Load Path - Beam to Column via Shear Plate
# Shows complete load transfer: Beam → Bolts → Plate → Welds → Column

import numpy as np
from steel_lib.load_path import LoadPathGenerator, LoadVector

# Create load path generator
generator = LoadPathGenerator()

# Get beam and column sections (already loaded)
beam = aisc['get_properties']('W18X35')
column = aisc['get_properties']('W14X90')

# Get a plate configuration (from previous example)
plate_config = {
    'thickness': 0.375,
    'width': 5.5,
    'length': 18.0,
    'area': 2.0625,
    'F_y': 50.0,
    'F_u': 65.0,
    'plate_grade': 'A572_50'
}

# Get a bolt configuration
bolt_config = {
    'bolt_size': 0.875,
    'bolt_grade': 'A325-N',
    'N_r': 4,
    'N_c': 1,
    'S_r': 3.0,
    'S_c': 3.0,
    'F_nv': 54.0,
    'F_nt': 90.0
}

# Define applied load (40 kip shear reaction)
applied_load = LoadVector(V_y=40.0)

# Create the load path
path = generator.create_simple_shear_connection(
    connection_id="SC001",
    beam_section=beam,
    column_section=column,
    plate_config=plate_config,
    bolt_config=bolt_config,
    applied_load=applied_load,
    eccentricity=3.0,       # 3" eccentricity
    weld_size=0.3125        # 5/16" fillet weld to column
)

print("LOAD PATH CREATED")
print("=" * 70)
print(f"Connection: {path.source_member} → {path.target_member}")
print(f"Applied Load: V = {path.applied_load.V_y:.1f} kips")
print(f"Eccentricity: {path.eccentricity:.2f} in")
print(f"Connection Type: {path.connection_type.name}")
print()

print("LOAD PATH ELEMENTS:")
print("-" * 70)
for i, elem in enumerate(path.elements, 1):
    print(f"{i}. {elem.element_type.name:15s} → {elem.element_id}")
    print(f"   Connection: {elem.connection_type.name}")
    if elem.geometry:
        key_props = list(elem.geometry.keys())[:3]
        props_str = ", ".join([f"{k}={elem.geometry[k]}" for k in key_props])
        print(f"   Geometry: {props_str}")


LOAD PATH CREATED
Connection: BEAM → COLUMN
Applied Load: V = 40.0 kips
Eccentricity: 3.00 in
Connection Type: HYBRID

LOAD PATH ELEMENTS:
----------------------------------------------------------------------
1. MEMBER_A        → SC001_BEAM
   Connection: BOLTED
   Geometry: designation=None, d=17.700000762939453, tw=0.30000001192092896
2. BOLTS_A         → SC001_BOLTS_BEAM
   Connection: BOLTED
   Geometry: bolt_size=0.875, bolt_grade=A325-N, N_r=4
3. PLATE           → SC001_PLATE
   Connection: HYBRID
   Geometry: thickness=0.375, width=5.5, length=18.0
4. WELDS_B         → SC001_WELDS_COLUMN
   Connection: WELDED
   Geometry: weld_size=0.3125, weld_length=18.0, weld_type=FILLET
5. MEMBER_B        → SC001_COLUMN
   Connection: WELDED
   Geometry: designation=None, d=14.0, tw=0.4399999976158142


In [10]:
# Example 2: Evaluate the Load Path
# Check capacity at each element and find the critical (weakest) link

results = generator.evaluate_load_path(path)

print("LOAD PATH EVALUATION RESULTS")
print("=" * 70)
print(f"Overall Status: {'✓ ADEQUATE' if results['is_adequate'] else '✗ INADEQUATE'}")
print(f"Critical Element: {results['critical_element']}")
print(f"Governing Capacity: {results.get('governing_capacity', 0):.1f} kips")
print(f"Max Utilization (D/C): {results.get('max_utilization', 0):.3f}")
print()

print("ELEMENT-BY-ELEMENT BREAKDOWN:")
print("-" * 70)

for elem_id, elem_result in results['elements'].items():
    print(f"\n{elem_id}")
    print(f"  Type: {elem_result['element_type']}")
    print(f"  Demand: {elem_result.get('demand', 0):.1f} kips")
    print(f"  Capacity: {elem_result.get('governing_capacity', 0):.1f} kips")
    print(f"  Utilization: {elem_result.get('utilization', 0):.3f}")
    print(f"  Governing Limit State: {elem_result.get('governing_limit_state', 'N/A')}")
    
    # Show all limit states
    if 'capacities' in elem_result and elem_result['capacities']:
        print(f"  All Limit States:")
        for ls_name, ls_capacity in elem_result['capacities'].items():
            status = "←" if ls_name == elem_result.get('governing_limit_state') else ""
            print(f"    • {ls_name:20s}: {ls_capacity:6.1f} kips {status}")

print("\n" + "=" * 70)
print("LOAD PATH DIAGRAM:")
print("-" * 70)
for i, elem in enumerate(path.elements):
    indent = "  " * i
    print(f"{elem.element_type.name:15s}", end="")
    if elem.governing_capacity:
        util = elem.utilization if elem.utilization else 0
        status = "✓" if util <= 1.0 else "✗"
        print(f"  φR_n = {elem.governing_capacity:5.1f}k  D/C = {util:.2f}  {status}")
    else:
        print()
    if i < len(path.elements) - 1:
        print("      ↓")


LOAD PATH EVALUATION RESULTS
Overall Status: ✓ ADEQUATE
Critical Element: SC001_PLATE
Governing Capacity: 61.9 kips
Max Utilization (D/C): 0.646

ELEMENT-BY-ELEMENT BREAKDOWN:
----------------------------------------------------------------------

SC001_BEAM
  Type: MEMBER
  Demand: 40.0 kips
  Capacity: 159.3 kips
  Utilization: 0.251
  Governing Limit State: web_shear
  All Limit States:
    • web_shear           :  159.3 kips ←

SC001_BOLTS_BEAM
  Type: BOLTS
  Demand: 20.0 kips
  Capacity: 80.0 kips
  Utilization: 0.250
  Governing Limit State: block_shear
  All Limit States:
    • bolt_shear          :   97.4 kips 
    • bearing             :  100.0 kips 
    • block_shear         :   80.0 kips ←

SC001_PLATE
  Type: PLATE
  Demand: 40.0 kips
  Capacity: 61.9 kips
  Utilization: 0.646
  Governing Limit State: shear_yielding
  All Limit States:
    • shear_yielding      :   61.9 kips ←
    • shear_rupture       :   75.0 kips 
    • block_shear         :   85.0 kips 

SC001_WELDS_CO

In [11]:
# Example 3: Batch Evaluation - Multiple Load Paths for Optimization
# Generate many connection configurations and find the optimal one

from steel_lib.load_path import generate_load_paths_batch

print("BATCH LOAD PATH GENERATION")
print("=" * 70)

# Use existing design space
# Plates: already have 54 configurations
# Bolts: generate a few patterns

bolt_patterns = generate_bolt_configurations(
    bolt_size=np.array([0.75, 0.875], dtype=np.float64),
    bolt_grade_id=np.array([0], dtype=np.int64),
    member_a_BHT_id=np.array([0], dtype=np.int64),
    member_b_BHT_id=np.array([0], dtype=np.int64),
    N_r=np.array([3, 4], dtype=np.int64),
    S_r=np.array([3.0], dtype=np.float64),
    N_c=np.array([1], dtype=np.int64),
    S_c=np.array([3.0], dtype=np.float64),
    L_ev=np.array([1.25], dtype=np.float64),
    L_eh=np.array([1.5], dtype=np.float64),
    Ga=np.array([0.0], dtype=np.float64)
)

# Select just a few plates for this example (to keep it fast)
sample_plates = {
    key: val[:3] for key, val in shear_plates.items()  # First 3 plates
}

# Create mock beam sections dict
beam_sections = {
    'designations': np.array(['W18X35']),
    'd': np.array([beam['d']]),
    'tw': np.array([beam['tw']]),
    'bf': np.array([beam['bf']]),
    'tf': np.array([beam['tf']]),
    'count': 1
}

# Generate load paths for different load levels
load_levels = np.array([30.0, 40.0, 50.0])  # kips
eccentricities = np.array([3.0, 3.0, 3.0])  # inches

print(f"Design Space:")
print(f"  Beams: {beam_sections['count']}")
print(f"  Plates: {len(sample_plates['thickness'])}")
print(f"  Bolt Patterns: {len(bolt_patterns['bolt_size'])}")
print(f"  Load Levels: {len(load_levels)}")
print(f"  Total Combinations: {beam_sections['count'] * len(sample_plates['thickness']) * len(bolt_patterns['bolt_size']) * len(load_levels)}")
print()

# Generate all load paths
all_paths = generate_load_paths_batch(
    beam_sections=beam_sections,
    column_sections=beam_sections,  # Same sections for simplicity
    plate_configs=sample_plates,
    bolt_configs=bolt_patterns,
    applied_loads=load_levels,
    eccentricities=eccentricities
)

print(f"Generated {len(all_paths)} load path combinations")
print()

# Evaluate first 5 as examples
print("Evaluating sample load paths...")
evaluated_paths = []

for i, lp in enumerate(all_paths[:5]):
    result = generator.evaluate_load_path(lp)
    evaluated_paths.append({
        'path': lp,
        'result': result,
        'load': lp.applied_load.V_y,
        'plate_t': lp.get_element(LoadPathElement.PLATE).geometry['thickness'],
        'n_bolts': lp.get_element(LoadPathElement.BOLTS_A).geometry['n_bolts']
    })

# Print results
print(f"\nSample Results:")
print("-" * 70)
for i, ep in enumerate(evaluated_paths, 1):
    status = "✓" if ep['result']['is_adequate'] else "✗"
    print(f"{i}. Load={ep['load']:.0f}k, Plate={ep['plate_t']:.3f}\", Bolts={ep['n_bolts']:.0f}")
    print(f"   Capacity={ep['result'].get('governing_capacity', 0):.1f}k, D/C={ep['result'].get('max_utilization', 0):.2f} {status}")
    print(f"   Critical: {ep['result']['critical_element']}")

# Find adequate connections
adequate = [ep for ep in evaluated_paths if ep['result']['is_adequate']]
print(f"\nAdequate Connections: {len(adequate)}/{len(evaluated_paths)}")

if adequate:
    # Find most efficient (closest to D/C = 1.0)
    optimal = min(adequate, key=lambda x: abs(1.0 - x['result']['max_utilization']))
    print(f"\nMost Efficient Connection:")
    print(f"  Path ID: {optimal['path'].path_id}")
    print(f"  Load: {optimal['load']:.1f} kips")
    print(f"  Plate: {optimal['plate_t']:.3f}\" thick")
    print(f"  Bolts: {optimal['n_bolts']:.0f} bolts")
    print(f"  Utilization: {optimal['result']['max_utilization']:.3f}")
    print(f"  Capacity: {optimal['result']['governing_capacity']:.1f} kips")


BATCH LOAD PATH GENERATION
Design Space:
  Beams: 1
  Plates: 3
  Bolt Patterns: 4
  Load Levels: 3
  Total Combinations: 36

Generated 36 load path combinations

Evaluating sample load paths...


NameError: name 'LoadPathElement' is not defined

In [12]:
# Example 4: Load Path Concept Summary
# Understanding the system

print("LOAD PATH CONCEPT")
print("=" * 70)
print("""
A load path tracks how forces transfer through a connection:

                BEAM (W18X35)
                     │
                     │ V = 40 kips
                     ↓
              ┌──────────────┐
              │  BOLTS (4)   │ ← Check: Bolt shear, bearing, tearout
              └──────────────┘
                     │
                     ↓
              ┌──────────────┐
              │ PLATE (3/8") │ ← Check: Shear yield/rupture, block shear
              └──────────────┘
                     │
                     ↓
              ┌──────────────┐
              │ WELDS (5/16")│ ← Check: Weld capacity
              └──────────────┘
                     │
                     ↓
              COLUMN (W14X90)  ← Check: Web local effects

Each element is evaluated. The WEAKEST element governs the connection capacity.
""")

print("\nKEY CLASSES:")
print("-" * 70)
print("""
1. LoadVector - Represents forces (P, Vx, Vy) and moments (Mx, My, Mz)
   Example: LoadVector(V_y=40.0)  # 40 kip shear

2. ConnectionElement - Single element in path (bolt, plate, weld, member)
   Contains: geometry, material, load, capacities

3. LoadPath - Complete path from source to target member
   Contains: multiple ConnectionElements in sequence

4. LoadPathGenerator - Creates and evaluates load paths
   Methods: create_simple_shear_connection(), evaluate_load_path()
""")

print("\nUSAGE WORKFLOW:")
print("-" * 70)
print("""
Step 1: Define members (beam, column) using AISC selector
Step 2: Define connection elements (plate, bolts) using generators
Step 3: Define applied loads (LoadVector)
Step 4: Create load path (generator.create_simple_shear_connection)
Step 5: Evaluate load path (generator.evaluate_load_path)
Step 6: Review results - find critical element, check adequacy
""")

print("\nINTEGRATION WITH YOUR SYSTEM:")
print("-" * 70)
print("""
✓ Works with: section_properties (AISC selector)
✓ Works with: plate_generator (connection plates)
✓ Works with: generator_combination (bolt patterns)
✓ Ready for: aisc_14th limit state functions (currently using placeholders)
✓ Ready for: Numba batch evaluation (vectorized load path checks)
✓ Ready for: Optimization (find min weight for given capacity)
""")

print("\nBENEFITS:")
print("-" * 70)
print("""
1. Systematic - Every transfer point is checked
2. Traceable - Complete documentation of load flow
3. Optimizable - Generate thousands of paths, find best
4. Modular - Easy to add new connection types
5. Visual - Can generate load flow diagrams
""")


LOAD PATH CONCEPT

A load path tracks how forces transfer through a connection:

                BEAM (W18X35)
                     │
                     │ V = 40 kips
                     ↓
              ┌──────────────┐
              │  BOLTS (4)   │ ← Check: Bolt shear, bearing, tearout
              └──────────────┘
                     │
                     ↓
              ┌──────────────┐
              │ PLATE (3/8") │ ← Check: Shear yield/rupture, block shear
              └──────────────┘
                     │
                     ↓
              ┌──────────────┐
              │ WELDS (5/16")│ ← Check: Weld capacity
              └──────────────┘
                     │
                     ↓
              COLUMN (W14X90)  ← Check: Web local effects

Each element is evaluated. The WEAKEST element governs the connection capacity.


KEY CLASSES:
----------------------------------------------------------------------

1. LoadVector - Represents forces (P, Vx, Vy) and moments (Mx,

# Weld Generator System

The weld generator follows the same pattern as bolt and plate generators:
- Integer mappings for efficiency (weld types, electrode grades)
- Columnar NumPy arrays for Numba compatibility
- Auto-calculated properties (throat, capacities)
- Multiple specialized generators for different weld types

In [18]:
# Import weld generator (with reload for development)
import importlib
import steel_lib.weld_generator
importlib.reload(steel_lib.weld_generator)

from steel_lib.weld_generator import (
    generate_fillet_welds,
    generate_groove_welds,
    generate_plug_slot_welds,
    get_weld_mapping_info,
    calculate_weld_length_required,
    WELD_TYPE_MAP,
    ELECTRODE_MAP,
    STANDARD_FILLET_SIZES
)

# Show weld system mappings
print("WELD GENERATOR SYSTEM")
print("=" * 70)
weld_mappings = get_weld_mapping_info()

print("\nWeld Types:")
for weld_id, weld_type in weld_mappings['weld_types'].items():
    print(f"  {weld_id}: {weld_type}")

print("\nElectrode Grades:")
for elec_id, electrode in weld_mappings['electrodes'].items():
    props = weld_mappings['electrode_properties'][electrode]
    print(f"  {elec_id}: {electrode} - F_EXX={props['F_EXX']:.0f} ksi, F_w={props['F_w']:.1f} ksi")

print(f"\nStandard Fillet Sizes: {len(STANDARD_FILLET_SIZES)} options")
print(f"  Range: {STANDARD_FILLET_SIZES[0]:.4f}\" ({STANDARD_FILLET_SIZES[0]*16:.0f}/16\") to {STANDARD_FILLET_SIZES[-1]:.4f}\" ({STANDARD_FILLET_SIZES[-1]*16:.0f}/16\")")
print(f"  Common sizes: 3/16\", 1/4\", 5/16\", 3/8\", 1/2\"")

WELD GENERATOR SYSTEM

Weld Types:
  0: FILLET
  1: CJP
  2: PJP
  3: PLUG
  4: SLOT
  5: FILLET_BOTH
  6: FILLET_INTERMITTENT

Electrode Grades:
  0: E60XX - F_EXX=60 ksi, F_w=36.0 ksi
  1: E70XX - F_EXX=70 ksi, F_w=42.0 ksi
  2: E80XX - F_EXX=80 ksi, F_w=48.0 ksi
  3: E90XX - F_EXX=90 ksi, F_w=54.0 ksi
  4: E100XX - F_EXX=100 ksi, F_w=60.0 ksi
  5: E110XX - F_EXX=110 ksi, F_w=66.0 ksi

Standard Fillet Sizes: 12 options
  Range: 0.1875" (3/16") to 1.0000" (16/16")
  Common sizes: 3/16", 1/4", 5/16", 3/8", 1/2"


In [19]:
# Example 1: Generate Fillet Weld Configurations
print("EXAMPLE 1: FILLET WELD GENERATOR")
print("=" * 70)

# Generate combinations of fillet welds
# - E70XX electrode (most common)
# - 3 weld sizes: 1/4", 5/16", 3/8"
# - 3 weld lengths: 12", 18", 24"
# - Both single and double fillet welds

fillet_welds = generate_fillet_welds(
    electrode_id=[1],                      # E70XX
    weld_size=[0.25, 0.3125, 0.375],      # 1/4", 5/16", 3/8"
    weld_length=[12.0, 18.0, 24.0],       # 12", 18", 24"
    both_sides=[False, True]              # Single and double fillet
)

print(f"\nGenerated {len(fillet_welds['weld_size'])} fillet weld configurations")
print(f"Output keys: {list(fillet_welds.keys())}")

# Show first 10 configurations
print("\nFirst 10 Fillet Weld Configurations:")
print("-" * 70)
for i in range(min(10, len(fillet_welds['weld_size']))):
    sides = "both sides" if fillet_welds['both_sides'][i] else "one side"
    print(f"Weld {i+1:2d}: {fillet_welds['electrode'][i]} "
          f"{fillet_welds['weld_size'][i]:.4f}\" ({fillet_welds['weld_size'][i]*16:.0f}/16\") fillet, "
          f"{fillet_welds['weld_length'][i]:5.1f}\" long, {sides:10s}")
    print(f"          Throat = {fillet_welds['throat'][i]:.4f}\", "
          f"F_w = {fillet_welds['F_w'][i]:.1f} ksi")
    print(f"          φR_n = {fillet_welds['phi_R_n'][i]:6.1f} kips, "
          f"φ(strength/inch) = {fillet_welds['phi_strength_per_inch'][i]:.2f} k/in")
    print()

# Summary statistics
print("\nCapacity Range:")
print(f"  Min φR_n: {fillet_welds['phi_R_n'].min():.1f} kips")
print(f"  Max φR_n: {fillet_welds['phi_R_n'].max():.1f} kips")
print(f"  Avg φR_n: {fillet_welds['phi_R_n'].mean():.1f} kips")

EXAMPLE 1: FILLET WELD GENERATOR

Generated 18 fillet weld configurations
Output keys: ['weld_type_id', 'electrode_id', 'weld_size', 'weld_length', 'both_sides', 'weld_type', 'electrode', 'F_EXX', 'F_w', 'throat', 'min_size_required', 'typical_max_size', 'R_n', 'phi_R_n', 'strength_per_inch', 'phi_strength_per_inch']

First 10 Fillet Weld Configurations:
----------------------------------------------------------------------
Weld  1: E70XX 0.2500" (4/16") fillet,  12.0" long, one side  
          Throat = 0.1767", F_w = 42.0 ksi
          φR_n =   66.8 kips, φ(strength/inch) = 5.57 k/in

Weld  2: E70XX 0.2500" (4/16") fillet,  12.0" long, both sides
          Throat = 0.1767", F_w = 42.0 ksi
          φR_n =  133.6 kips, φ(strength/inch) = 5.57 k/in

Weld  3: E70XX 0.2500" (4/16") fillet,  18.0" long, one side  
          Throat = 0.1767", F_w = 42.0 ksi
          φR_n =  100.2 kips, φ(strength/inch) = 5.57 k/in

Weld  4: E70XX 0.2500" (4/16") fillet,  18.0" long, both sides
          T

In [15]:
# Example 2: Weld Length Calculator
print("EXAMPLE 2: WELD LENGTH CALCULATOR")
print("=" * 70)

# Scenario: Need to transfer 50 kip shear force
force_required = 50.0  # kips

# Try different weld sizes
weld_sizes_to_try = [0.1875, 0.25, 0.3125, 0.375, 0.5]  # 3/16", 1/4", 5/16", 3/8", 1/2"

print(f"\nFor {force_required:.0f} kip force with E70XX electrode:")
print("-" * 70)
print(f"{'Weld Size':>15s} {'Single Side':>15s} {'Both Sides':>15s}")
print("-" * 70)

for size in weld_sizes_to_try:
    L_single = calculate_weld_length_required(
        force=force_required,
        electrode_grade='E70XX',
        weld_size=size,
        both_sides=False
    )
    
    L_double = calculate_weld_length_required(
        force=force_required,
        electrode_grade='E70XX',
        weld_size=size,
        both_sides=True
    )
    
    size_str = f"{size:.4f}\" ({size*16:.0f}/16\")"
    print(f"{size_str:>15s} {L_single:>13.1f}\" {L_double:>13.1f}\"")

print("\nNote: Add end returns and minimum length requirements per AISC")

EXAMPLE 2: WELD LENGTH CALCULATOR

For 50 kip force with E70XX electrode:
----------------------------------------------------------------------
      Weld Size     Single Side      Both Sides
----------------------------------------------------------------------
0.1875" (3/16")          12.0"           6.0"
0.2500" (4/16")           9.0"           4.5"
0.3125" (5/16")           7.2"           3.6"
0.3750" (6/16")           6.0"           3.0"
0.5000" (8/16")           4.5"           2.2"

Note: Add end returns and minimum length requirements per AISC


In [16]:
# Example 3: Complete Connection - Plates and Welds
print("EXAMPLE 3: INTEGRATED PLATE + WELD SYSTEM")
print("=" * 70)

# Scenario: Shear connection with welds attaching plate to column
# Transfer 40 kip shear force

print("\nConnection Type: Shear plate welded to column")
print("Design Force: 40 kips shear")

# Step 1: Generate shear plate configurations
from steel_lib.plate_generator import generate_shear_plates
plate_config = generate_shear_plates(
    plate_grade_id=[1],             # A572-50
    thickness=[0.375, 0.5],         # 3/8", 1/2"
    width=[5.0, 6.0],               # 5", 6"
    length=[15.0, 18.0, 21.0]       # 15", 18", 21"
)

print(f"\nGenerated {len(plate_config['thickness'])} plate configurations")

# Step 2: Generate weld configuration
weld_config = generate_fillet_welds(
    electrode_id=[1],                    # E70XX
    weld_size=[0.25, 0.3125, 0.375],    # 1/4", 5/16", 3/8"
    weld_length=[15.0, 18.0, 21.0],     # Match plate length
    both_sides=[True]                    # Both sides of plate
)

print(f"Generated {len(weld_config['weld_size'])} weld configurations")

# Step 3: Find configurations that work for 40 kip force
print("\n" + "=" * 70)
print("VIABLE CONFIGURATIONS FOR 40 KIP FORCE")
print("=" * 70)

force_required = 40.0  # kips
viable_count = 0

for plate_idx in range(len(plate_config['thickness'])):
    plate_capacity = plate_config['phi_V_n_gross'][plate_idx]
    
    # Only look for matching weld lengths
    for weld_idx in range(len(weld_config['weld_size'])):
        if abs(weld_config['weld_length'][weld_idx] - plate_config['length'][plate_idx]) < 0.1:
            weld_capacity = weld_config['phi_R_n'][weld_idx]
            
            # Check if combination works
            min_capacity = min(plate_capacity, weld_capacity)
            
            if min_capacity >= force_required and viable_count < 5:  # Show first 5
                viable_count += 1
                utilization = force_required / min_capacity * 100
                
                print(f"\nConfiguration {viable_count}:")
                print(f"  PLATE: {plate_config['plate_grade'][plate_idx]} "
                      f"PL {plate_config['width'][plate_idx]:.1f}\" x "
                      f"{plate_config['thickness'][plate_idx]:.3f}\" x "
                      f"{plate_config['length'][plate_idx]:.1f}\"")
                print(f"    φV_n = {plate_capacity:.1f} kips")
                
                print(f"  WELD: {weld_config['electrode'][weld_idx]} "
                      f"{weld_config['weld_size'][weld_idx]*16:.0f}/16\" fillet, "
                      f"{weld_config['weld_length'][weld_idx]:.1f}\" both sides")
                print(f"    Throat: {weld_config['throat'][weld_idx]:.4f}\", "
                      f"φR_n = {weld_capacity:.1f} kips")
                
                print(f"  CAPACITY: φR_n = {min_capacity:.1f} kips, "
                      f"Utilization = {utilization:.1f}%", end="")
                
                if min_capacity == plate_capacity:
                    print(f" (Limited by PLATE)")
                else:
                    print(f" (Limited by WELD)")

print(f"\n\nTotal viable configurations found: {viable_count}+")
print("Note: Bolt capacity also needs to be checked for beam side of connection")

EXAMPLE 3: INTEGRATED PLATE + WELD SYSTEM

Connection Type: Shear plate welded to column
Design Force: 40 kips shear

Generated 12 plate configurations
Generated 9 weld configurations

VIABLE CONFIGURATIONS FOR 40 KIP FORCE

Configuration 1:
  PLATE: A572_50 PL 5.0" x 0.375" x 15.0"
    φV_n = 56.2 kips
  WELD: E70XX 4/16" fillet, 15.0" both sides
    Throat: 0.1767", φR_n = 167.0 kips
  CAPACITY: φR_n = 56.2 kips, Utilization = 71.1% (Limited by PLATE)

Configuration 2:
  PLATE: A572_50 PL 5.0" x 0.375" x 15.0"
    φV_n = 56.2 kips
  WELD: E70XX 5/16" fillet, 15.0" both sides
    Throat: 0.2209", φR_n = 208.8 kips
  CAPACITY: φR_n = 56.2 kips, Utilization = 71.1% (Limited by PLATE)

Configuration 3:
  PLATE: A572_50 PL 5.0" x 0.375" x 15.0"
    φV_n = 56.2 kips
  WELD: E70XX 6/16" fillet, 15.0" both sides
    Throat: 0.2651", φR_n = 250.5 kips
  CAPACITY: φR_n = 56.2 kips, Utilization = 71.1% (Limited by PLATE)

Configuration 4:
  PLATE: A572_50 PL 5.0" x 0.375" x 18.0"
    φV_n = 56.

## Complete System: Sections + Bolts + Plates + Welds

The complete steel connection design toolkit is now available!

In [17]:
# Example 4: Complete Load Path with Weld Generator Integration
print("EXAMPLE 4: LOAD PATH WITH WELD GENERATOR")
print("=" * 70)

# Reload modules to get latest changes
import importlib
import steel_lib.load_path
importlib.reload(steel_lib.load_path)

from steel_lib.load_path import LoadPathGenerator, LoadVector

# Scenario: W18x35 beam to W14x90 column shear connection
# - 40 kip shear force
# - Bolted to beam, welded to column

print("\nConnection: W18x35 beam to W14x90 column")
print("Load: 40 kips shear")
print("Configuration: Bolted to beam web, welded to column flange")

# Step 1: Generate weld configuration
print("\n" + "-" * 70)
print("Step 1: Generate Weld Configuration")
print("-" * 70)

from steel_lib.weld_generator import generate_fillet_welds

# Generate weld options for 18" plate
welds = generate_fillet_welds(
    electrode_id=[1],                # E70XX
    weld_size=[0.25, 0.3125],       # 1/4", 5/16"
    weld_length=[18.0],             # 18" to match plate
    both_sides=[True]                # Both sides of plate
)

print(f"Generated {len(welds['weld_size'])} weld configurations:")
for i in range(len(welds['weld_size'])):
    print(f"  Weld {i+1}: {welds['electrode'][i]} "
          f"{welds['weld_size'][i]*16:.0f}/16\" fillet, "
          f"{welds['weld_length'][i]:.1f}\" both sides")
    print(f"    Throat: {welds['throat'][i]:.4f}\", "
          f"φR_n = {welds['phi_R_n'][i]:.1f} kips")

# Step 2: Create load path with weld config
print("\n" + "-" * 70)
print("Step 2: Create Load Path with Weld Integration")
print("-" * 70)

generator = LoadPathGenerator()

# Applied load
applied_load = LoadVector(V_y=40.0)

# Mock beam/column sections (simplified)
beam_section = {
    'designation': 'W18X35',
    'd': 17.70,
    'tw': 0.300,
    'bf': 6.00,
    'tf': 0.425,
    'F_y': 50.0,
    'F_u': 65.0
}

column_section = {
    'designation': 'W14X90',
    'd': 14.02,
    'tw': 0.440,
    'bf': 14.52,
    'tf': 0.710,
    'F_y': 50.0,
    'F_u': 65.0
}

# Mock plate config
plate_config = {
    'thickness': 0.375,
    'width': 5.0,
    'length': 18.0,
    'area': 0.375 * 5.0,
    'F_y': 50.0,
    'F_u': 65.0,
    'plate_grade': 'A572_50'
}

# Mock bolt config
bolt_config = {
    'bolt_size': 0.75,
    'bolt_grade': 'A325',
    'N_r': 5,
    'N_c': 2,
    'S_r': 3.0,
    'S_c': 3.0,
    'F_nv': 48.0,
    'F_nt': 90.0
}

# Select first weld configuration (1/4" weld)
weld_idx = 0
weld_config_selected = {
    'weld_type': welds['weld_type'][weld_idx],
    'electrode': welds['electrode'][weld_idx],
    'weld_size': welds['weld_size'][weld_idx],
    'weld_length': welds['weld_length'][weld_idx],
    'throat': welds['throat'][weld_idx],
    'both_sides': True,
    'F_EXX': welds['F_EXX'][weld_idx],
    'F_w': welds['F_w'][weld_idx],
    'R_n': welds['R_n'][weld_idx],
    'phi_R_n': welds['phi_R_n'][weld_idx],
    'strength_per_inch': welds['strength_per_inch'][weld_idx],
    'phi_strength_per_inch': welds['phi_strength_per_inch'][weld_idx]
}

# Create load path WITH weld_config parameter
path = generator.create_simple_shear_connection(
    connection_id='SHEAR_CONN_01',
    beam_section=beam_section,
    column_section=column_section,
    plate_config=plate_config,
    bolt_config=bolt_config,
    applied_load=applied_load,
    eccentricity=3.0,
    weld_config=weld_config_selected  # ← Using weld generator output!
)

print(f"\nLoad path created: {path.path_id}")
print(f"Connection type: {path.connection_type.name}")
print(f"Elements: {len(path.elements)}")
for elem in path.elements:
    print(f"  - {elem.element_id} ({elem.element_type.name})")

# Step 3: Evaluate load path
print("\n" + "-" * 70)
print("Step 3: Evaluate Load Path")
print("-" * 70)

results = generator.evaluate_load_path(path)

print(f"\nEvaluation Results:")
print(f"  Is Adequate: {results['is_adequate']}")
print(f"  Critical Element: {results['critical_element']}")
print(f"  Governing Capacity: {results['governing_capacity']:.1f} kips")
print(f"  Max Utilization: {results['max_utilization']*100:.1f}%")

# Show detailed weld evaluation
print("\n" + "-" * 70)
print("Weld Element Details")
print("-" * 70)

weld_results = results['elements']['SHEAR_CONN_01_WELDS_COLUMN']
print(f"\nWeld Type: {weld_results['weld_info']['weld_type']}")
print(f"Electrode: {welds['electrode'][weld_idx]}")
print(f"Size: {weld_results['weld_info']['weld_size']:.4f}\" ({weld_results['weld_info']['weld_size']*16:.0f}/16\")")
print(f"Length: {weld_results['weld_info']['weld_length']:.1f}\"")
print(f"Throat: {weld_results['weld_info']['throat']:.4f}\"")
print(f"F_w: {weld_results['weld_info']['F_w']:.1f} ksi")
print(f"Both Sides: {weld_results['weld_info']['both_sides']}")
print(f"Effective Area: {weld_results['weld_info']['effective_area']:.2f} in²")

print(f"\nCapacities:")
for limit_state, capacity in weld_results['capacities'].items():
    marker = "←" if limit_state == weld_results['governing_limit_state'] else ""
    print(f"  {limit_state}: {capacity:.1f} kips {marker}")

print(f"\nDemand: {weld_results['demand']:.1f} kips")
print(f"Utilization: {weld_results['utilization']*100:.1f}%")

print("\n" + "=" * 70)
print("✓ Load path system now fully integrated with weld generator!")

EXAMPLE 4: LOAD PATH WITH WELD GENERATOR

Connection: W18x35 beam to W14x90 column
Load: 40 kips shear
Configuration: Bolted to beam web, welded to column flange

----------------------------------------------------------------------
Step 1: Generate Weld Configuration
----------------------------------------------------------------------
Generated 2 weld configurations:
  Weld 1: E70XX 4/16" fillet, 18.0" both sides
    Throat: 0.1767", φR_n = 200.4 kips
  Weld 2: E70XX 5/16" fillet, 18.0" both sides
    Throat: 0.2209", φR_n = 250.5 kips

----------------------------------------------------------------------
Step 2: Create Load Path with Weld Integration
----------------------------------------------------------------------

Load path created: SHEAR_CONN_01
Connection type: BOLTED
Elements: 5
  - SHEAR_CONN_01_BEAM (MEMBER_A)
  - SHEAR_CONN_01_BOLTS_BEAM (BOLTS_A)
  - SHEAR_CONN_01_PLATE (PLATE)
  - SHEAR_CONN_01_WELDS_COLUMN (WELDS_B)
  - SHEAR_CONN_01_COLUMN (MEMBER_B)

-----------

## BATCH PROCESSING: The Real Power

The system is designed for batch processing - evaluate THOUSANDS of configurations simultaneously!

## Example 7: Batch Section + Material Selection for Multiple Connections

In [ ]:
# Example 7: Batch Section + Material Selection for Multiple Connections
print("EXAMPLE 7: BATCH SECTION + MATERIAL SELECTION")
print("=" * 80)
print("Evaluating multiple sections with multiple materials")
print("Demonstrates efficient batch processing of section × material combinations")
print("=" * 80)

import time
import numpy as np
from steel_lib.section_properties import create_aisc_section_selector
from steel_lib.load_path import LoadPathGenerator, LoadVector
from steel_lib.weld_generator import generate_fillet_welds
from steel_lib.plate_generator import generate_shear_plates
from steel_lib.generator_combination import generate_bolt_configurations

# Load AISC database
print("\n1. Loading AISC Section Database...")
aisc = create_aisc_section_selector('aisc-shapes-database-v16.0.xlsx')
print(f"   ✓ Loaded {aisc['database_info']['n_sections']} sections")

# Define design scenario: Compare multiple beam-column combinations with different materials
print("\n" + "▶" * 40)
print("SCENARIO: Multi-Section, Multi-Material Connection Design")
print("▶" * 40)

# We want to evaluate these beam sections
beam_designations = ['W18X35', 'W21X44', 'W24X55', 'W18X50']
# With these material options
material_options = ['A36', 'A992', 'A572_50']
# Applied load
applied_load_kips = 50.0

print(f"\nDesign Parameters:")
print(f"  Beam sections: {', '.join(beam_designations)} ({len(beam_designations)} options)")
print(f"  Materials: {', '.join(material_options)} ({len(material_options)} options)")
print(f"  Applied load: {applied_load_kips:.1f} kips")
print(f"  Total beam combinations: {len(beam_designations)} × {len(material_options)} = {len(beam_designations) * len(material_options)}")

# Step 1: Get all beam section × material combinations using batch method
print("\n" + "-" * 80)
print("Step 1: Generate Section × Material Combinations (BATCH)")
print("-" * 80)

start = time.time()
beam_combos = aisc['get_sections_batch'](beam_designations, material_options)
batch_time = (time.time() - start) * 1000

print(f"✓ Generated {beam_combos['count']} combinations in {batch_time:.2f} ms")
print(f"  Time per combination: {batch_time/beam_combos['count']:.3f} ms")
print()

# Display the combinations
print("   Beam Section × Material Combinations:")
print(f"   {'#':<4} {'Designation':<12} {'Material':<10} {'F_y (ksi)':<12} {'d (in)':<10} {'tw (in)':<10} {'W (lb/ft)':<10}")
print(f"   {'-'*4} {'-'*12} {'-'*10} {'-'*12} {'-'*10} {'-'*10} {'-'*10}")
for i in range(beam_combos['count']):
    print(f"   {i+1:<4} {beam_combos['designations'][i]:<12} "
          f"{beam_combos['materials'][i]:<10} "
          f"{beam_combos['F_y'][i]:<12.1f} "
          f"{beam_combos['d'][i]:<10.2f} "
          f"{beam_combos['tw'][i]:<10.3f} "
          f"{beam_combos['W'][i]:<10.1f}")

# Step 2: For each beam combo, generate connection components
print("\n" + "-" * 80)
print("Step 2: Generate Connection Components for Each Beam")
print("-" * 80)

# Column selection (keep simple - same for all)
column_designation = 'W14X90'
column = aisc['get_section_with_material'](column_designation, material='A992')
print(f"\nColumn (common for all): {column_designation} + {column['material']}")
print(f"  d = {column['d']:.2f} in, tw = {column['tw']:.3f} in, F_y = {column['F_y']:.1f} ksi")

# Generate connection components (simplified - fewer options for speed)
print("\nGenerating connection components...")

plates = generate_shear_plates(
    plate_grade_id=[1],                 # A572_50
    thickness=[0.375, 0.5],             # 3/8", 1/2"
    width=[5.0],                        # 5"
    length=[12.0, 15.0]                 # 12", 15"
)

bolts = generate_bolt_configurations(
    bolt_size=np.array([0.75], dtype=np.float64),
    bolt_grade_id=np.array([0], dtype=np.int64),
    member_a_BHT_id=np.array([0], dtype=np.int64),
    member_b_BHT_id=np.array([0], dtype=np.int64),
    N_r=np.array([4, 5], dtype=np.int64),
    S_r=np.array([3.0], dtype=np.float64),
    N_c=np.array([1], dtype=np.int64),
    S_c=np.array([3.0], dtype=np.float64),
    L_ev=np.array([1.5], dtype=np.float64),
    L_eh=np.array([1.5], dtype=np.float64),
    Ga=np.array([3.0], dtype=np.float64)
)

welds = generate_fillet_welds(
    electrode_id=[1],                   # E70XX
    weld_size=[0.1875, 0.25],          # 3/16", 1/4"
    weld_length=[12.0, 15.0],
    both_sides=[True]
)

print(f"  ✓ {len(plates['thickness'])} plate configs")
print(f"  ✓ {len(bolts['bolt_size'])} bolt patterns")
print(f"  ✓ {len(welds['weld_size'])} weld configs")

total_configs_per_beam = len(plates['thickness']) * len(bolts['bolt_size']) * len(welds['weld_size'])
print(f"  Total configs per beam: {total_configs_per_beam}")
print(f"  Total configs all beams: {beam_combos['count']} × {total_configs_per_beam} = {beam_combos['count'] * total_configs_per_beam}")

# Step 3: Evaluate each beam combo with all connection configurations
print("\n" + "-" * 80)
print("Step 3: Evaluate All Combinations (VECTORIZED)")
print("-" * 80)

applied_load = LoadVector(V_x=0.0, V_y=applied_load_kips, V_z=0.0, M_x=0.0, M_y=0.0, M_z=0.0)

results_summary = []

eval_start = time.time()

for beam_idx in range(beam_combos['count']):
    # Extract beam properties for this combination
    beam_section = {
        'designation': beam_combos['designations'][beam_idx],
        'material': beam_combos['materials'][beam_idx],
        'd': beam_combos['d'][beam_idx],
        'tw': beam_combos['tw'][beam_idx],
        'bf': beam_combos['bf'][beam_idx],
        'tf': beam_combos['tf'][beam_idx],
        'F_y': beam_combos['F_y'][beam_idx],
        'F_u': beam_combos['F_u'][beam_idx],
    }
    
    # Create load path generator for this beam
    generator = LoadPathGenerator()
    
    # Connection: Beam web → Plate (weld) → Plate (bolt) → Column web
    generator.beam_web_to_plate_weld(
        beam_props=beam_section,
        plate_props_dict=plates,
        weld_props_dict=welds
    )
    
    generator.plate_to_column_web_bolted(
        plate_props_dict=plates,
        bolt_props_dict=bolts,
        column_props=column
    )
    
    # Get all load paths and evaluate
    all_paths = generator.get_all_paths()
    
    # Evaluate all paths
    viable_count = 0
    best_utilization = float('inf')
    best_config = None
    
    for path in all_paths:
        result = path.evaluate_capacity(applied_load)
        utilization = result['max_utilization']
        
        if result['is_adequate']:
            viable_count += 1
            if utilization < best_utilization:
                best_utilization = utilization
                best_config = {
                    'plate_thickness': path.elements[0].plate_props['thickness'],
                    'weld_size': path.elements[0].weld_props['weld_size'],
                    'bolt_pattern': f"{path.elements[1].bolt_props['N_r']}x{path.elements[1].bolt_props['N_c']}",
                    'utilization': utilization
                }
    
    results_summary.append({
        'beam': beam_section['designation'],
        'material': beam_section['material'],
        'F_y': beam_section['F_y'],
        'viable_designs': viable_count,
        'total_evaluated': len(all_paths),
        'best_utilization': best_utilization if best_config else None,
        'best_config': best_config
    })

eval_time = (time.time() - eval_start) * 1000

print(f"\n✓ Evaluated all combinations in {eval_time:.1f} ms")
print(f"  Total paths evaluated: {sum(r['total_evaluated'] for r in results_summary)}")
print(f"  Average time per beam combo: {eval_time / beam_combos['count']:.1f} ms")

# Step 4: Display results
print("\n" + "-" * 80)
print("Step 4: Results Summary")
print("-" * 80)

print(f"\n{'Beam':<12} {'Material':<10} {'F_y':<8} {'Viable':<10} {'Total':<10} {'Best Util':<12} {'Best Config'}")
print(f"{'-'*12} {'-'*10} {'-'*8} {'-'*10} {'-'*10} {'-'*12} {'-'*30}")

for r in results_summary:
    util_str = f"{r['best_utilization']:.1%}" if r['best_utilization'] else "N/A"
    config_str = ""
    if r['best_config']:
        config_str = f"t={r['best_config']['plate_thickness']:.3f}\", w={r['best_config']['weld_size']:.3f}\", {r['best_config']['bolt_pattern']}"
    else:
        config_str = "No viable design"
    
    print(f"{r['beam']:<12} {r['material']:<10} {r['F_y']:<8.1f} "
          f"{r['viable_designs']:<10} {r['total_evaluated']:<10} "
          f"{util_str:<12} {config_str}")

# Find overall best design (lowest utilization across all beam-material combos)
best_overall = min((r for r in results_summary if r['best_config']), 
                   key=lambda x: x['best_utilization'])

print("\n" + "=" * 80)
print("OPTIMAL DESIGN IDENTIFIED:")
print("=" * 80)
print(f"  Beam: {best_overall['beam']} + {best_overall['material']} (F_y = {best_overall['F_y']:.1f} ksi)")
print(f"  Plate: {best_overall['best_config']['plate_thickness']:.3f}\" thick")
print(f"  Weld: {best_overall['best_config']['weld_size']:.3f}\" fillet")
print(f"  Bolts: {best_overall['best_config']['bolt_pattern']} pattern")
print(f"  Utilization: {best_overall['best_utilization']:.1%}")
print(f"  Viable alternatives: {best_overall['viable_designs']}")
print("=" * 80)

print(f"\n⚡ PERFORMANCE SUMMARY:")
print(f"  Section batch generation: {batch_time:.2f} ms")
print(f"  Load path evaluation: {eval_time:.1f} ms")
print(f"  Total designs evaluated: {sum(r['total_evaluated'] for r in results_summary)}")
print(f"  Time per design: {eval_time / sum(r['total_evaluated'] for r in results_summary):.3f} ms")

## AISC Section Selector - Enhanced with Materials

The section selector now includes material properties and multiple access methods!

In [ ]:
# Demonstration: Enhanced AISC Section Selector with Materials
print("ENHANCED AISC SECTION SELECTOR")
print("=" * 80)

from steel_lib.section_properties import create_aisc_section_selector
import numpy as np

# Load the selector
print("\n1. Loading AISC Database...")
aisc = create_aisc_section_selector('aisc-shapes-database-v16.0.xlsx')
print(f"   ✓ Loaded {aisc['database_info']['n_sections']} sections")
print(f"   ✓ Supported materials: {', '.join(aisc['database_info']['materials'])}")

# Method 1: Direct property lookup (fast, single section)
print("\n" + "-" * 80)
print("2. Method 1: get_properties() - Fast single section lookup")
print("-" * 80)

section = aisc['get_properties']('W18X35', material='A992')
if section:
    print(f"   W18X35 with A992 steel:")
    print(f"   - d = {section['d']:.2f} in")
    print(f"   - tw = {section['tw']:.3f} in")
    print(f"   - F_y = {section['F_y']:.1f} ksi")
    print(f"   - F_u = {section['F_u']:.1f} ksi")
    print(f"   - Material: {section['material']}")

# Method 2: Using select_by_properties (flexible, can filter)
print("\n" + "-" * 80)
print("3. Method 2: get_section_with_material() - Flexible filtering approach")
print("-" * 80)

section_a36 = aisc['get_section_with_material']('W18X35', material='A36')
section_a992 = aisc['get_section_with_material']('W18X35', material='A992')

print(f"   W18X35 comparison:")
print(f"   A36:  F_y = {section_a36['F_y']:.1f} ksi, F_u = {section_a36['F_u']:.1f} ksi")
print(f"   A992: F_y = {section_a992['F_y']:.1f} ksi, F_u = {section_a992['F_u']:.1f} ksi")

# Method 3: Using select_by_properties for multiple sections
print("\n" + "-" * 80)
print("4. Method 3: select_by_properties() - Batch filtering")
print("-" * 80)

# Get multiple sections at once
results = aisc['select_by_properties']({
    "designation": ["W18X35", "W21X44", "W24X55"]
})

print(f"   Found {results['count']} sections:")
for i in range(results['count']):
    print(f"   - {results['designations'][i]}: "
          f"d={results['d'][i]:.1f}\", "
          f"tw={results['tw'][i]:.3f}\", "
          f"W={results['W'][i]:.1f} lb/ft")

# Method 4: Filter by properties (not just designation)
print("\n" + "-" * 80)
print("5. Method 4: Filter by properties - Find sections meeting criteria")
print("-" * 80)

# Find W shapes with depth between 18-22" and weight under 50 lb/ft
results = aisc['select_by_properties']({
    "type": ["W"],
    "d": {"min": 18.0, "max": 22.0},
    "W": {"max": 50.0}
})

print(f"   Found {results['count']} W shapes (18-22\" deep, <50 lb/ft):")
for i in range(min(5, results['count'])):  # Show first 5
    print(f"   {i+1}. {results['designations'][i]}: "
          f"d={results['d'][i]:.1f}\", "
          f"W={results['W'][i]:.1f} lb/ft")

# Method 5: Complete workflow - Get section with material for design
print("\n" + "-" * 80)
print("6. Complete Workflow - Section Selection for Design")
print("-" * 80)

# Find lightest W18 beam
results = aisc['select_by_properties']({
    "type": ["W"],
    "d": {"min": 17.0, "max": 19.0}
})

# Sort by weight
sorted_idx = np.argsort(results['W'])
lightest = results['designations'][sorted_idx[0]]

print(f"   Lightest W18: {lightest}")

# Get full properties with material
beam = aisc['get_section_with_material'](lightest, material='A992')
print(f"   Properties for design:")
print(f"   - Designation: {beam['designation']}")
print(f"   - Weight: {beam['W']:.1f} lb/ft")
print(f"   - Depth: {beam['d']:.2f} in")
print(f"   - Web thickness: {beam['tw']:.3f} in")
print(f"   - Material: {beam['material']}")
print(f"   - F_y: {beam['F_y']:.1f} ksi")
print(f"   - F_u: {beam['F_u']:.1f} ksi")

# Method 6: BATCH - Multiple sections with multiple materials
print("\n" + "-" * 80)
print("7. Method 6: get_sections_batch() - Multiple Sections × Multiple Materials")
print("-" * 80)

# Get 3 sections with 3 materials = 9 combinations
sections_to_try = ['W18X35', 'W21X44', 'W24X55']
materials_to_try = ['A36', 'A992', 'A588']

batch_results = aisc['get_sections_batch'](sections_to_try, materials_to_try)

print(f"   Generated {batch_results['count']} combinations:")
print(f"   {len(sections_to_try)} sections × {len(materials_to_try)} materials = {batch_results['count']}")
print()

# Display in table format
print(f"   {'Section':<12} {'Material':<10} {'F_y (ksi)':<12} {'F_u (ksi)':<12} {'d (in)':<10} {'W (lb/ft)':<10}")
print(f"   {'-'*12} {'-'*10} {'-'*12} {'-'*12} {'-'*10} {'-'*10}")

for i in range(batch_results['count']):
    print(f"   {batch_results['designations'][i]:<12} "
          f"{batch_results['materials'][i]:<10} "
          f"{batch_results['F_y'][i]:<12.1f} "
          f"{batch_results['F_u'][i]:<12.1f} "
          f"{batch_results['d'][i]:<10.2f} "
          f"{batch_results['W'][i]:<10.1f}")

# Demonstrate vectorized operations on batch results
print("\n   Vectorized analysis on all combinations:")
print(f"   - F_y range: {batch_results['F_y'].min():.1f} to {batch_results['F_y'].max():.1f} ksi")
print(f"   - Weight range: {batch_results['W'].min():.1f} to {batch_results['W'].max():.1f} lb/ft")
print(f"   - Depth range: {batch_results['d'].min():.1f} to {batch_results['d'].max():.1f} in")

# Find optimal combination (highest F_y, lightest weight)
strength_weight_ratio = batch_results['F_y'] / batch_results['W']
best_idx = np.argmax(strength_weight_ratio)

print(f"\n   Optimal strength-to-weight ratio:")
print(f"   ✓ {batch_results['designations'][best_idx]} + {batch_results['materials'][best_idx]}")
print(f"     F_y = {batch_results['F_y'][best_idx]:.1f} ksi, "
      f"W = {batch_results['W'][best_idx]:.1f} lb/ft, "
      f"ratio = {strength_weight_ratio[best_idx]:.3f}")

print("\n" + "=" * 80)
print("✓ Multiple flexible methods for section selection!")
print("  1. get_properties() - Fast single section")
print("  2. get_section_with_material() - Includes material properties")
print("  3. select_by_properties() - Flexible filtering by any property")
print("  4. get_sections_batch() - Multiple sections × materials (VECTORIZED)")
print("=" * 80)

In [ ]:
# Example 5: BATCH PROCESSING - Multiple Configurations Simultaneously
print("EXAMPLE 5: BATCH LOAD PATH EVALUATION")
print("=" * 80)
print("Evaluating MULTIPLE connection designs in parallel")
print("This demonstrates the true power of vectorized operations!")
print("=" * 80)

import time
import numpy as np
from steel_lib.load_path import LoadPathGenerator, LoadVector
from steel_lib.weld_generator import generate_fillet_welds
from steel_lib.plate_generator import generate_shear_plates
from steel_lib.generator_combination import generate_bolt_configurations

# Scenario: Design shear connections for 5 different beam-column pairs
# We'll generate all possible combinations and find optimal designs

print("\n" + "▶" * 40)
print("SCENARIO: Shear Connections for Multiple Beam-Column Pairs")
print("▶" * 40)

# Define 5 beam-column pairs with different loads
beam_column_pairs = [
    {'beam': 'W18x35', 'column': 'W14x90', 'load': 40.0, 'connection_id': 'CONN_A'},
    {'beam': 'W21x44', 'column': 'W14x120', 'load': 55.0, 'connection_id': 'CONN_B'},
    {'beam': 'W24x55', 'column': 'W14x145', 'load': 70.0, 'connection_id': 'CONN_C'},
    {'beam': 'W16x26', 'column': 'W14x68', 'load': 30.0, 'connection_id': 'CONN_D'},
    {'beam': 'W18x50', 'column': 'W14x109', 'load': 60.0, 'connection_id': 'CONN_E'},
]

print(f"\nConnection Pairs to Design: {len(beam_column_pairs)}")
for pair in beam_column_pairs:
    print(f"  {pair['connection_id']}: {pair['beam']} to {pair['column']} @ {pair['load']:.0f} kips")

# Step 1: Generate all possible plate configurations (batch)
print("\n" + "-" * 80)
print("Step 1: Generate Plate Configurations (Batch)")
print("-" * 80)

start_time = time.time()

plates = generate_shear_plates(
    plate_grade_id=[0, 1],              # A36, A572_50
    thickness=[0.375, 0.5, 0.625],      # 3/8", 1/2", 5/8"
    width=[4.0, 5.0, 6.0],              # 4", 5", 6"
    length=[12.0, 15.0, 18.0, 21.0]     # Various lengths
)

plate_gen_time = (time.time() - start_time) * 1000
print(f"✓ Generated {len(plates['thickness'])} plate configurations in {plate_gen_time:.2f} ms")
print(f"  Grade options: {len([0, 1])}")
print(f"  Thickness options: {len([0.375, 0.5, 0.625])}")
print(f"  Width options: {len([4.0, 5.0, 6.0])}")
print(f"  Length options: {len([12.0, 15.0, 18.0, 21.0])}")
print(f"  Total combinations: {len(plates['thickness'])} plates")

# Step 2: Generate all possible bolt patterns (batch)
print("\n" + "-" * 80)
print("Step 2: Generate Bolt Patterns (Batch)")
print("-" * 80)

start_time = time.time()

bolts = generate_bolt_configurations(
    bolt_size=np.array([0.75, 0.875], dtype=np.float64),     # 3/4", 7/8"
    bolt_grade_id=np.array([0, 1], dtype=np.int64),          # A325-N, A325-X
    member_a_BHT_id=np.array([0], dtype=np.int64),           # STD holes
    member_b_BHT_id=np.array([0], dtype=np.int64),           # STD holes
    N_r=np.array([3, 4, 5, 6], dtype=np.int64),              # 3-6 rows
    S_r=np.array([3.0], dtype=np.float64),                   # 3" row spacing
    N_c=np.array([1, 2], dtype=np.int64),                    # 1-2 columns
    S_c=np.array([3.0], dtype=np.float64),                   # 3" column spacing
    L_ev=np.array([1.5], dtype=np.float64),                  # 1.5" edge distance
    L_eh=np.array([1.5], dtype=np.float64),                  # 1.5" edge distance
    Ga=np.array([3.0], dtype=np.float64)                     # 3" gauge
)

bolt_gen_time = (time.time() - start_time) * 1000
print(f"✓ Generated {len(bolts['bolt_size'])} bolt patterns in {bolt_gen_time:.2f} ms")
print(f"  Bolt grades: {len([0, 1])}")
print(f"  Diameters: {len([0.75, 0.875])}")
print(f"  Row options: {len([3, 4, 5, 6])}")
print(f"  Column options: {len([1, 2])}")
print(f"  Total combinations: {len(bolts['bolt_size'])} patterns")

# Step 3: Generate all possible weld configurations (batch)
print("\n" + "-" * 80)
print("Step 3: Generate Weld Configurations (Batch)")
print("-" * 80)

start_time = time.time()

welds = generate_fillet_welds(
    electrode_id=[1, 2],                # E70XX, E80XX
    weld_size=[0.1875, 0.25, 0.3125, 0.375],  # 3/16" to 3/8"
    weld_length=[12.0, 15.0, 18.0, 21.0],     # Match plate lengths
    both_sides=[True]
)

weld_gen_time = (time.time() - start_time) * 1000
print(f"✓ Generated {len(welds['weld_size'])} weld configurations in {weld_gen_time:.2f} ms")
print(f"  Electrode grades: {len([1, 2])}")
print(f"  Weld sizes: {len([0.1875, 0.25, 0.3125, 0.375])}")
print(f"  Lengths: {len([12.0, 15.0, 18.0, 21.0])}")
print(f"  Total combinations: {len(welds['weld_size'])} welds")

# Step 4: Create and evaluate load paths for all combinations
print("\n" + "-" * 80)
print("Step 4: Batch Evaluate Load Paths")
print("-" * 80)

generator = LoadPathGenerator()

# Load AISC section database
from steel_lib.section_properties import create_aisc_section_selector

print("\nLoading AISC section database...")
aisc = create_aisc_section_selector('aisc-shapes-database-v16.0.xlsx')
print(f"✓ Loaded {aisc['database_info']['n_sections']} sections")
print(f"  Supported materials: {', '.join(aisc['database_info']['materials'])}")

start_time = time.time()

all_paths = []
viable_designs = {pair['connection_id']: [] for pair in beam_column_pairs}

# For each connection pair
for pair in beam_column_pairs:
    conn_id = pair['connection_id']
    applied_load = LoadVector(V_y=pair['load'])
    
    # Get section properties using select_by_properties approach
    # More flexible - can filter by designation or other properties
    beam_designation = pair['beam'].upper()  # W18X35 format
    column_designation = pair['column'].upper()
    
    # Use get_section_with_material for cleaner code with material properties
    beam_section = aisc['get_section_with_material'](beam_designation, material='A992')
    column_section = aisc['get_section_with_material'](column_designation, material='A992')
    
    # Skip if sections not found
    if beam_section is None or column_section is None:
        print(f"  Warning: Section not found for {pair['beam']} or {pair['column']}, skipping...")
        continue
    
    # Try multiple plate/bolt/weld combinations (sampling for demo)
    # In real system, you'd evaluate ALL combinations
    for plate_idx in range(0, min(20, len(plates['thickness']))):
        for bolt_idx in range(0, min(10, len(bolts['bolt_size']))):
            for weld_idx in range(0, min(5, len(welds['weld_size']))):
                
                # Create plate config from batch
                plate_config = {
                    'thickness': plates['thickness'][plate_idx],
                    'width': plates['width'][plate_idx],
                    'length': plates['length'][plate_idx],
                    'area': plates['area'][plate_idx],
                    'F_y': plates['F_y'][plate_idx],
                    'F_u': plates['F_u'][plate_idx],
                    'plate_grade': plates['plate_grade'][plate_idx]
                }
                
                # Create bolt config from batch
                bolt_config = {
                    'bolt_size': bolts['bolt_size'][bolt_idx],
                    'bolt_grade': bolts['bolt_grade'][bolt_idx],
                    'N_r': int(bolts['N_r'][bolt_idx]),
                    'N_c': int(bolts['N_c'][bolt_idx]),
                    'S_r': bolts['S_r'][bolt_idx],
                    'S_c': bolts['S_c'][bolt_idx],
                    'F_nv': bolts['F_nv'][bolt_idx],
                    'F_nt': bolts['F_nt'][bolt_idx]
                }
                
                # Create weld config from batch
                weld_config = {
                    'weld_type': welds['weld_type'][weld_idx],
                    'electrode': welds['electrode'][weld_idx],
                    'weld_size': welds['weld_size'][weld_idx],
                    'weld_length': welds['weld_length'][weld_idx],
                    'throat': welds['throat'][weld_idx],
                    'both_sides': True,
                    'F_EXX': welds['F_EXX'][weld_idx],
                    'F_w': welds['F_w'][weld_idx],
                    'R_n': welds['R_n'][weld_idx],
                    'phi_R_n': welds['phi_R_n'][weld_idx],
                    'strength_per_inch': welds['strength_per_inch'][weld_idx],
                    'phi_strength_per_inch': welds['phi_strength_per_inch'][weld_idx]
                }
                
                # Create load path
                try:
                    path = generator.create_simple_shear_connection(
                        connection_id=f"{conn_id}_P{plate_idx}_B{bolt_idx}_W{weld_idx}",
                        beam_section=beam_section,
                        column_section=column_section,
                        plate_config=plate_config,
                        bolt_config=bolt_config,
                        applied_load=applied_load,
                        eccentricity=3.0,
                        weld_config=weld_config
                    )
                    
                    # Evaluate
                    result = generator.evaluate_load_path(path)
                    
                    # Store if adequate
                    if result['is_adequate']:
                        viable_designs[conn_id].append({
                            'path': path,
                            'result': result,
                            'plate_idx': plate_idx,
                            'bolt_idx': bolt_idx,
                            'weld_idx': weld_idx,
                            'capacity': result['governing_capacity'],
                            'utilization': result['max_utilization'],
                            'plate_config': plate_config,
                            'bolt_config': bolt_config,
                            'weld_config': weld_config
                        })
                    
                    all_paths.append((path, result))
                    
                except Exception as e:
                    # Skip invalid combinations
                    continue

batch_eval_time = (time.time() - start_time) * 1000

print(f"✓ Evaluated {len(all_paths)} load paths in {batch_eval_time:.1f} ms")
if len(all_paths) > 0:
    print(f"  Average time per path: {batch_eval_time/len(all_paths):.3f} ms")
else:
    print(f"  No valid paths created - check section properties format")

# Step 5: Analyze results
print("\n" + "=" * 80)
print("BATCH EVALUATION RESULTS")
print("=" * 80)

for pair in beam_column_pairs:
    conn_id = pair['connection_id']
    viable_count = len(viable_designs[conn_id])
    
    print(f"\n{conn_id}: {pair['beam']} to {pair['column']} @ {pair['load']:.0f} kips")
    print(f"  Viable designs found: {viable_count}")
    
    if viable_count > 0:
        # Find most efficient design (lowest utilization)
        best_design = min(viable_designs[conn_id], key=lambda x: x['utilization'])
        
        print(f"  ✓ OPTIMAL DESIGN:")
        print(f"    Plate: {best_design['plate_config']['plate_grade']} "
              f"PL {best_design['plate_config']['width']:.1f}\" x "
              f"{best_design['plate_config']['thickness']:.3f}\" x "
              f"{best_design['plate_config']['length']:.1f}\"")
        print(f"    Bolts: {best_design['bolt_config']['bolt_grade']} "
              f"{best_design['bolt_config']['bolt_size']:.3f}\" "
              f"({best_design['bolt_config']['N_r']}x{best_design['bolt_config']['N_c']})")
        print(f"    Welds: {best_design['weld_config']['electrode']} "
              f"{best_design['weld_config']['weld_size']*16:.0f}/16\" fillet, "
              f"{best_design['weld_config']['weld_length']:.1f}\" both sides")
        print(f"    Capacity: {best_design['capacity']:.1f} kips")
        print(f"    Utilization: {best_design['utilization']*100:.1f}%")
        print(f"    Critical Element: {best_design['result']['critical_element']}")
    else:
        print(f"  ✗ No viable designs found in sample set")

# Summary statistics
print("\n" + "=" * 80)
print("PERFORMANCE SUMMARY")
print("=" * 80)
print(f"Total configurations evaluated: {len(all_paths)}")
print(f"Total viable designs: {sum(len(v) for v in viable_designs.values())}")
print(f"Total generation time: {plate_gen_time + bolt_gen_time + weld_gen_time:.1f} ms")
print(f"Total evaluation time: {batch_eval_time:.1f} ms")
if len(all_paths) > 0:
    print(f"Average per config: {(batch_eval_time/len(all_paths)):.3f} ms")
    print(f"\n✓ BATCH PROCESSING enables rapid design space exploration!")
    print("  Real-world: Evaluate 10,000+ combinations in seconds")
else:
    print("\n✗ No configurations were successfully evaluated")
    print("  Check that section designations are correct (e.g., 'W18x35' not 'W18X35')")

EXAMPLE 5: BATCH LOAD PATH EVALUATION
Evaluating MULTIPLE connection designs in parallel
This demonstrates the true power of vectorized operations!

▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶
SCENARIO: Shear Connections for Multiple Beam-Column Pairs
▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶▶

Connection Pairs to Design: 5
  CONN_A: W18x35 to W14x90 @ 40 kips
  CONN_B: W21x44 to W14x120 @ 55 kips
  CONN_C: W24x55 to W14x145 @ 70 kips
  CONN_D: W16x26 to W14x68 @ 30 kips
  CONN_E: W18x50 to W14x109 @ 60 kips

--------------------------------------------------------------------------------
Step 1: Generate Plate Configurations (Batch)
--------------------------------------------------------------------------------
✓ Generated 72 plate configurations in 0.00 ms
  Grade options: 2
  Thickness options: 3
  Width options: 3
  Length options: 4
  Total combinations: 72 plates

--------------------------------------------------------------------------------
Step 2: Generate Bolt Patterns (Batch)
-

In [28]:
# Example 6: Simplified Batch Comparison
# Show side-by-side comparison of different designs for same load

print("EXAMPLE 6: BATCH COMPARISON - Find Optimal Design")
print("=" * 80)
print("Goal: Design a 50 kip shear connection - compare multiple options")
print("=" * 80)

from steel_lib.load_path import LoadPathGenerator, LoadVector
from steel_lib.weld_generator import generate_fillet_welds
from steel_lib.plate_generator import generate_shear_plates
from steel_lib.section_properties import create_aisc_section_selector
import time

# Load AISC section database
print("\nLoading AISC section database...")
aisc = create_aisc_section_selector('aisc-shapes-database-v16.0.xlsx')
print(f"✓ Loaded {aisc['database_info']['n_sections']} sections")
print(f"  Supported materials: {', '.join(aisc['database_info']['materials'])}")

# Get section properties using select_by_properties approach
# More flexible - can specify material directly
beam_section = aisc['get_section_with_material']('W21X44', material='A992')
column_section = aisc['get_section_with_material']('W14X120', material='A992')

# Check if sections were found
if beam_section is None or column_section is None:
    print(f"\n✗ ERROR: Sections not found in database")
    print(f"  Beam 'W21X44' found: {beam_section is not None}")
    print(f"  Column 'W14X120' found: {column_section is not None}")
else:
    # Sections already include F_y, F_u, and designation from get_section_with_material
    pass

applied_load = LoadVector(V_y=50.0)

print(f"\nBeam: {beam_section['designation']}")
print(f"Column: {column_section['designation']}")
print(f"Applied Load: {applied_load.V_y:.0f} kips shear")

# Generate design alternatives
print("\n" + "-" * 80)
print("Generating Design Alternatives (Batch)")
print("-" * 80)

start = time.time()

# Generate 3 different plate strategies
plates_light = generate_shear_plates(
    plate_grade_id=[1],          # A572_50
    thickness=[0.375],           # 3/8"
    width=[5.0],
    length=[18.0, 21.0]          # Longer plates
)

plates_medium = generate_shear_plates(
    plate_grade_id=[1],          # A572_50
    thickness=[0.5],             # 1/2"
    width=[5.0],
    length=[15.0, 18.0]          # Medium plates
)

plates_heavy = generate_shear_plates(
    plate_grade_id=[1],          # A572_50
    thickness=[0.625],           # 5/8"
    width=[5.0],
    length=[12.0, 15.0]          # Shorter, thicker
)

# Generate 3 different weld strategies
welds_light = generate_fillet_welds(
    electrode_id=[1],            # E70XX
    weld_size=[0.1875],          # 3/16"
    weld_length=[18.0, 21.0],
    both_sides=[True]
)

welds_medium = generate_fillet_welds(
    electrode_id=[1],            # E70XX
    weld_size=[0.25],            # 1/4"
    weld_length=[15.0, 18.0],
    both_sides=[True]
)

welds_heavy = generate_fillet_welds(
    electrode_id=[1],            # E70XX
    weld_size=[0.3125],          # 5/16"
    weld_length=[12.0, 15.0],
    both_sides=[True]
)

gen_time = (time.time() - start) * 1000

print(f"✓ Generated 3 plate strategies: {len(plates_light['thickness']) + len(plates_medium['thickness']) + len(plates_heavy['thickness'])} total plates")
print(f"✓ Generated 3 weld strategies: {len(welds_light['weld_size']) + len(welds_medium['weld_size']) + len(welds_heavy['weld_size'])} total welds")
print(f"  Generation time: {gen_time:.2f} ms")

# Mock bolt config (simplified for this example)
bolt_config_base = {
    'bolt_size': 0.875,
    'bolt_grade': 'a325_x',
    'N_r': 6,
    'N_c': 2,
    'S_r': 3.0,
    'S_c': 3.0,
    'F_nv': 48.0,
    'F_nt': 90.0
}

# Evaluate all strategies
print("\n" + "-" * 80)
print("Evaluating Strategies")
print("-" * 80)

generator = LoadPathGenerator()
strategies = []

start = time.time()

# Strategy 1: Light plate + light weld
for p_idx in range(len(plates_light['thickness'])):
    for w_idx in range(len(welds_light['weld_size'])):
        try:
            plate_cfg = {
                'thickness': plates_light['thickness'][p_idx],
                'width': plates_light['width'][p_idx],
                'length': plates_light['length'][p_idx],
                'area': plates_light['area'][p_idx],
                'F_y': plates_light['F_y'][p_idx],
                'F_u': plates_light['F_u'][p_idx],
                'plate_grade': plates_light['plate_grade'][p_idx]
            }
            
            weld_cfg = {
                'weld_type': welds_light['weld_type'][w_idx],
                'electrode': welds_light['electrode'][w_idx],
                'weld_size': welds_light['weld_size'][w_idx],
                'weld_length': welds_light['weld_length'][w_idx],
                'throat': welds_light['throat'][w_idx],
                'both_sides': True,
                'F_EXX': welds_light['F_EXX'][w_idx],
                'F_w': welds_light['F_w'][w_idx],
                'R_n': welds_light['R_n'][w_idx],
                'phi_R_n': welds_light['phi_R_n'][w_idx],
                'strength_per_inch': welds_light['strength_per_inch'][w_idx],
                'phi_strength_per_inch': welds_light['phi_strength_per_inch'][w_idx]
            }
            
            path = generator.create_simple_shear_connection(
                connection_id=f'LIGHT_{p_idx}_{w_idx}',
                beam_section=beam_section,
                column_section=column_section,
                plate_config=plate_cfg,
                bolt_config=bolt_config_base,
                applied_load=applied_load,
                eccentricity=3.0,
                weld_config=weld_cfg
            )
            
            result = generator.evaluate_load_path(path)
            
            if result['is_adequate']:
                strategies.append({
                    'name': 'Light (3/8" plate + 3/16" weld)',
                    'plate': plate_cfg,
                    'weld': weld_cfg,
                    'capacity': result['governing_capacity'],
                    'utilization': result['max_utilization'],
                    'weight_estimate': plates_light['total_weight'][p_idx],
                    'result': result
                })
        except:
            pass

# Strategy 2: Medium plate + medium weld
for p_idx in range(len(plates_medium['thickness'])):
    for w_idx in range(len(welds_medium['weld_size'])):
        try:
            plate_cfg = {
                'thickness': plates_medium['thickness'][p_idx],
                'width': plates_medium['width'][p_idx],
                'length': plates_medium['length'][p_idx],
                'area': plates_medium['area'][p_idx],
                'F_y': plates_medium['F_y'][p_idx],
                'F_u': plates_medium['F_u'][p_idx],
                'plate_grade': plates_medium['plate_grade'][p_idx]
            }
            
            weld_cfg = {
                'weld_type': welds_medium['weld_type'][w_idx],
                'electrode': welds_medium['electrode'][w_idx],
                'weld_size': welds_medium['weld_size'][w_idx],
                'weld_length': welds_medium['weld_length'][w_idx],
                'throat': welds_medium['throat'][w_idx],
                'both_sides': True,
                'F_EXX': welds_medium['F_EXX'][w_idx],
                'F_w': welds_medium['F_w'][w_idx],
                'R_n': welds_medium['R_n'][w_idx],
                'phi_R_n': welds_medium['phi_R_n'][w_idx],
                'strength_per_inch': welds_medium['strength_per_inch'][w_idx],
                'phi_strength_per_inch': welds_medium['phi_strength_per_inch'][w_idx]
            }
            
            path = generator.create_simple_shear_connection(
                connection_id=f'MEDIUM_{p_idx}_{w_idx}',
                beam_section=beam_section,
                column_section=column_section,
                plate_config=plate_cfg,
                bolt_config=bolt_config_base,
                applied_load=applied_load,
                eccentricity=3.0,
                weld_config=weld_cfg
            )
            
            result = generator.evaluate_load_path(path)
            
            if result['is_adequate']:
                strategies.append({
                    'name': 'Medium (1/2" plate + 1/4" weld)',
                    'plate': plate_cfg,
                    'weld': weld_cfg,
                    'capacity': result['governing_capacity'],
                    'utilization': result['max_utilization'],
                    'weight_estimate': plates_medium['total_weight'][p_idx],
                    'result': result
                })
        except:
            pass

# Strategy 3: Heavy plate + heavy weld
for p_idx in range(len(plates_heavy['thickness'])):
    for w_idx in range(len(welds_heavy['weld_size'])):
        try:
            plate_cfg = {
                'thickness': plates_heavy['thickness'][p_idx],
                'width': plates_heavy['width'][p_idx],
                'length': plates_heavy['length'][p_idx],
                'area': plates_heavy['area'][p_idx],
                'F_y': plates_heavy['F_y'][p_idx],
                'F_u': plates_heavy['F_u'][p_idx],
                'plate_grade': plates_heavy['plate_grade'][p_idx]
            }
            
            weld_cfg = {
                'weld_type': welds_heavy['weld_type'][w_idx],
                'electrode': welds_heavy['electrode'][w_idx],
                'weld_size': welds_heavy['weld_size'][w_idx],
                'weld_length': welds_heavy['weld_length'][w_idx],
                'throat': welds_heavy['throat'][w_idx],
                'both_sides': True,
                'F_EXX': welds_heavy['F_EXX'][w_idx],
                'F_w': welds_heavy['F_w'][w_idx],
                'R_n': welds_heavy['R_n'][w_idx],
                'phi_R_n': welds_heavy['phi_R_n'][w_idx],
                'strength_per_inch': welds_heavy['strength_per_inch'][w_idx],
                'phi_strength_per_inch': welds_heavy['phi_strength_per_inch'][w_idx]
            }
            
            path = generator.create_simple_shear_connection(
                connection_id=f'HEAVY_{p_idx}_{w_idx}',
                beam_section=beam_section,
                column_section=column_section,
                plate_config=plate_cfg,
                bolt_config=bolt_config_base,
                applied_load=applied_load,
                eccentricity=3.0,
                weld_config=weld_cfg
            )
            
            result = generator.evaluate_load_path(path)
            
            if result['is_adequate']:
                strategies.append({
                    'name': 'Heavy (5/8" plate + 5/16" weld)',
                    'plate': plate_cfg,
                    'weld': weld_cfg,
                    'capacity': result['governing_capacity'],
                    'utilization': result['max_utilization'],
                    'weight_estimate': plates_heavy['total_weight'][p_idx],
                    'result': result
                })
        except:
            pass

eval_time = (time.time() - start) * 1000

print(f"✓ Evaluated {len(strategies)} viable strategies in {eval_time:.1f} ms")

# Compare results
print("\n" + "=" * 80)
print("STRATEGY COMPARISON - Ranked by Efficiency")
print("=" * 80)

# Sort by utilization (higher is more efficient)
strategies_sorted = sorted(strategies, key=lambda x: x['utilization'], reverse=True)

for i, strat in enumerate(strategies_sorted[:5], 1):  # Show top 5
    print(f"\n{i}. {strat['name']}")
    print(f"   Plate: {strat['plate']['plate_grade']} "
          f"PL {strat['plate']['width']:.1f}\" x "
          f"{strat['plate']['thickness']:.3f}\" x "
          f"{strat['plate']['length']:.1f}\"")
    print(f"   Weld: {strat['weld']['electrode']} "
          f"{strat['weld']['weld_size']*16:.0f}/16\" fillet, "
          f"{strat['weld']['weld_length']:.1f}\" both sides")
    print(f"   Capacity: {strat['capacity']:.1f} kips")
    print(f"   Utilization: {strat['utilization']*100:.1f}% ← {'OPTIMAL' if i == 1 else ''}")
    print(f"   Weight: {strat['weight_estimate']:.1f} lb")
    print(f"   Critical: {strat['result']['critical_element']}")

print("\n" + "=" * 80)
print("✓ Batch evaluation enables intelligent design optimization!")
print(f"  Total time: {gen_time + eval_time:.1f} ms")

EXAMPLE 6: BATCH COMPARISON - Find Optimal Design
Goal: Design a 50 kip shear connection - compare multiple options

Loading AISC section database...
Loading AISC Shapes Database...
Loaded 2299 sections from AISC database
✓ Loaded 2299 sections

Beam: W21X44
Column: W14X120
Applied Load: 50 kips shear

--------------------------------------------------------------------------------
Generating Design Alternatives (Batch)
--------------------------------------------------------------------------------
✓ Generated 3 plate strategies: 6 total plates
✓ Generated 3 weld strategies: 6 total welds
  Generation time: 0.00 ms

--------------------------------------------------------------------------------
Evaluating Strategies
--------------------------------------------------------------------------------
✓ Evaluated 12 viable strategies in 1.0 ms

STRATEGY COMPARISON - Ranked by Efficiency

1. Light (3/8" plate + 3/16" weld)
   Plate: A572_50 PL 5.0" x 0.375" x 18.0"
   Weld: E70XX 3/16" fill

In [1]:
from steel_lib.section_properties import create_aisc_section_selector
aisc_16th = create_aisc_section_selector('aisc-shapes-database-v16.0.xlsx')

Loading AISC Shapes Database...
Loaded 2299 sections from AISC database


In [7]:
from steel_lib.plate_generator import generate_shear_plates
from steel_lib.weld_generator import generate_fillet_welds
from steel_lib.section_properties import create_aisc_section_selector
from steel_lib.generator_combination import generate_bolt_configurations
import numpy as np 

# Select beam section with material properties (default A992)
beam = aisc_16th['select_by_properties']({
    "designation": ["W14X53"],
}, material='A992')

# Can also filter by other properties and specify different material
# beam_a36 = aisc_16th['select_by_properties']({
#     "designation": ["W14X53"],
# }, material='A36')

# Generate plates - using 't' for thickness per variable naming protocol
plates = generate_shear_plates(
    plate_grade_id=[0, 1],  # 0=A36, 1=A572_50
    t=[0.25, 0.375],
    a=[1, 2]
)

# Generate welds - integer IDs only
welds = generate_fillet_welds(
    electrode_id=[1, 2],  # 1=E70XX, 2=E80XX
    weld_size=[0.1875, 0.25],
    weld_length=[12.0, 15.0],
    both_sides=[True, False]
)

# Generate bolts - integer IDs only
bolts = generate_bolt_configurations(
    bolt_size=np.array([0.75, 0.875], dtype=np.float64),
    bolt_grade_id=np.array([0, 1], dtype=np.int64),  # 0=a325_n, 1=a325_x
    member_a_BHT_id=np.array([0], dtype=np.int64),   # 0=STD
    member_b_BHT_id=np.array([0], dtype=np.int64),   # 0=STD
    N_r=np.array([3, 4], dtype=np.int64),
    S_r=np.array([3.0], dtype=np.float64),
    N_c=np.array([1, 2], dtype=np.int64),
    S_c=np.array([3.0], dtype=np.float64),
    L_ev=np.array([1.5], dtype=np.float64),
    L_eh=np.array([1.5], dtype=np.float64),
    Ga=np.array([3.0], dtype=np.float64)
)

# Show results with material properties
print("Beam properties (with material):")
print(f"  Designation: {beam['designations'][0]}")
print(f"  F_y: {beam['F_y'][0]} ksi")
print(f"  F_u: {beam['F_u'][0]} ksi")
print(f"  d: {beam['d'][0]:.2f} in")
print(f"  bf: {beam['bf'][0]:.2f} in")

print(f"\nGenerated {len(plates['t'])} plate configurations")
print(f"Generated {len(welds['weld_size'])} weld configurations") 
print(f"Generated {len(bolts['bolt_size'])} bolt configurations")



Beam properties (with material):
  Designation: W14X53
  F_y: 50.0 ksi
  F_u: 65.0 ksi
  d: 13.90 in
  bf: 8.06 in

Generated 8 plate configurations
Generated 16 weld configurations
Generated 16 bolt configurations


In [8]:
class load_transfer:
    def __init__(self, source, target, load_vector):
        self.source = source
        self.target = target
        self.load_vector = load_vector
    

    

In [9]:
# Reload module to get latest changes
import importlib
import steel_lib.load_transfer
importlib.reload(steel_lib.load_transfer)
from steel_lib.load_transfer import (
    create_connection_matrix,
    generate_connection_combinations,
    chain_connections,
    extract_connection_properties,
    apply_eccentricity_moments
)
print("Module reloaded")

Module reloaded


In [10]:
# Debug - check the connection matrices
print("plate_column_conn details:")
print(f"  n_connections: {plate_column_conn['n_connections']}")
print(f"  member_a_idx (plates): {plate_column_conn['member_a_idx'][:10]}")
print(f"  member_b_idx (columns): {plate_column_conn['member_b_idx'][:10]}")
print(f"  connector_idx (welds): {plate_column_conn['connector_idx'][:10]}")
print(f"\nColumns array size: {len(columns['designations'])}")
print(f"Max column index in plate_column_conn: {plate_column_conn['member_b_idx'].max()}")


plate_column_conn details:


NameError: name 'plate_column_conn' is not defined

In [11]:
# Import generator functions
from steel_lib.plate_generator import generate_shear_plates
from steel_lib.weld_generator import generate_fillet_welds
from steel_lib.generator_combination import generate_bolt_configurations
import numpy as np

# Import load transfer functions
from steel_lib.load_transfer import (
    create_connection_matrix,
    generate_connection_combinations,
    chain_connections,
    extract_connection_properties,
    apply_eccentricity_moments
)

print("Load Transfer System - Using Real Generator Data")
print("=" * 70)

print("\n1. Generate configurations using actual generators")

# Generate beam sections using AISC selector
beams = aisc_16th['select_by_properties']({
    "designation": ["W14X53", "W14X68"],
}, material='A992')
print(f"   Beams: {len(beams['designations'])} configurations")
for i in range(len(beams['designations'])):
    print(f"      {beams['designations'][i]} - tw={beams['tw'][i]:.3f}\", F_y={beams['F_y'][i]} ksi")

# Generate plates using plate generator
plates = generate_shear_plates(
    plate_grade_id=[0, 1],  # A36, A572_50
    t=[0.25, 0.375, 0.5],
    a=[1, 2]
)
print(f"   Plates: {len(plates['t'])} configurations")
print(f"      t={plates['t'][:5]} in, F_y={plates['F_y'][:5]} ksi")

# Generate bolts using bolt generator
bolts = generate_bolt_configurations(
    bolt_size=np.array([0.75, 0.875], dtype=np.float64),
    bolt_grade_id=np.array([0, 1], dtype=np.int64),  # a325_n, a325_x
    member_a_BHT_id=np.array([0], dtype=np.int64),
    member_b_BHT_id=np.array([0], dtype=np.int64),
    N_r=np.array([3, 4], dtype=np.int64),
    S_r=np.array([3.0], dtype=np.float64),
    N_c=np.array([1, 2], dtype=np.int64),
    S_c=np.array([3.0], dtype=np.float64),
    L_ev=np.array([1.5], dtype=np.float64),
    L_eh=np.array([1.5], dtype=np.float64),
    Ga=np.array([3.0], dtype=np.float64)
)
print(f"   Bolts: {len(bolts['bolt_size'])} configurations")
print(f"      First 5: N_r={bolts['N_r'][:5]}, size={bolts['bolt_size'][:5]}\"")

# Generate welds using weld generator
welds = generate_fillet_welds(
    electrode_id=[1, 2],  # E70XX, E80XX
    weld_size=[0.1875, 0.25, 0.3125],
    weld_length=[12.0, 15.0],
    both_sides=[True, False]
)
print(f"   Welds: {len(welds['weld_size'])} configurations")
print(f"      First 5: size={welds['weld_size'][:5]}\", F_w={welds['F_w'][:5]} ksi")

# Generate column sections using AISC selector
columns = aisc_16th['select_by_properties']({
    "designation": ["W14X90"],
}, material='A992')
print(f"   Columns: {len(columns['designations'])} configurations")
print(f"      {columns['designations'][0]} - tw={columns['tw'][0]:.3f}\", F_y={columns['F_y'][0]} ksi")

print("\n2. Create connection matrices (relating configs)")

# Connection 1: Beam -> Bolts -> Plate
beam_plate_conn = generate_connection_combinations(
    member_a_count=len(beams['designations']),
    member_b_count=len(plates['t']),
    connector_count=len(bolts['bolt_size'])
)
print(f"   Beam-Plate connections: {beam_plate_conn['n_connections']} combinations")

# Connection 2: Plate -> Welds -> Column
plate_column_conn = generate_connection_combinations(
    member_a_count=len(plates['t']),
    member_b_count=len(columns['designations']),
    connector_count=len(welds['weld_size'])
)
print(f"   Plate-Column connections: {plate_column_conn['n_connections']} combinations")

print("\n3. Chain connections (Beam -> Plate -> Column)")

# Chain them together - NOW SUPPORTS MULTIPLE CONNECTIONS!
full_chain = chain_connections(beam_plate_conn, plate_column_conn)
print(f"   Full chains: {full_chain['n_chains']} valid combinations")
print(f"   Chain structure: {full_chain['n_members']} members, {full_chain['n_connectors']} connectors")

print("\n4. Show first 3 connection chains")

# Show first few chains using new indexed format
for idx in range(min(3, full_chain['n_chains'])):
    m2_idx = full_chain['member_2_idx'][idx]
    print(f"\n   Chain {idx}:")
    print(f"     Member 0 (Beam)[{full_chain['member_0_idx'][idx]}]: {beams['designations'][full_chain['member_0_idx'][idx]]}")
    print(f"     → Connector 0 (Bolts)[{full_chain['connector_0_idx'][idx]}]: {bolts['N_r'][full_chain['connector_0_idx'][idx]]} rows × {bolts['bolt_size'][full_chain['connector_0_idx'][idx]]}\" dia")
    print(f"     → Member 1 (Plate)[{full_chain['member_1_idx'][idx]}]: t={plates['t'][full_chain['member_1_idx'][idx]]}\" (F_y={plates['F_y'][full_chain['member_1_idx'][idx]]} ksi)")
    print(f"     → Connector 1 (Welds)[{full_chain['connector_1_idx'][idx]}]: size={welds['weld_size'][full_chain['connector_1_idx'][idx]]}\"")
    print(f"     → Member 2 (Column)[{m2_idx}]: {columns['designations'][m2_idx] if m2_idx < len(columns['designations']) else 'INDEX OUT OF BOUNDS!'}")

print("\n5. Extract properties for ALL interfaces (multi-interface extraction)")

# NEW SYNTAX: Pass data in order: member0, connector0, member1, connector1, member2
# Specify which properties to extract from each interface
props = extract_connection_properties(
    full_chain,
    beams, bolts, plates, welds, columns,  # Data in order!
    properties_to_extract={
        'member_0': ['tw', 'F_y', 'F_u'],           # Beam properties
        'connector_0': ['bolt_size', 'N_r', 'F_nv'], # Bolt properties
        'member_1': ['t', 'F_y', 'F_u'],             # Plate properties
        'connector_1': ['weld_size', 'F_w'],         # Weld properties
        'member_2': ['tw', 'F_y', 'F_u']             # Column properties
    }
)

print(f"   Extracted properties from ALL interfaces (first 3 chains):")
print(f"     Beam (m0) tw: {props['m0_tw'][:3]}")
print(f"     Bolt (c0) N_r: {props['c0_N_r'][:3]}")
print(f"     Plate (m1) t: {props['m1_t'][:3]}")
print(f"     Weld (c1) size: {props['c1_weld_size'][:3]}")
print(f"     Column (m2) tw: {props['m2_tw'][:3]}")

print("\n6. Apply loads and eccentricity")

# Create loads for all chains
n_chains = full_chain['n_chains']
loads = {
    'P': np.full(n_chains, 10.0),
    'V_y': np.full(n_chains, 40.0),
    'V_z': np.zeros(n_chains),
    'M_x': np.zeros(n_chains),
    'M_y': np.zeros(n_chains),
    'M_z': np.zeros(n_chains)
}

# Apply eccentricity (2" offset)
eccentricity = {'e_x': np.full(n_chains, 2.0)}
loads_with_ecc = apply_eccentricity_moments(loads, eccentricity)

print(f"   Applied P={loads['P'][0]} kips, V_y={loads['V_y'][0]} kips")
print(f"   With e_x=2.0\", resulting M_z: {loads_with_ecc['M_z'][:3]} kip-in")

print("\n" + "=" * 70)
print("✓ Ready for limit state checks at ALL interfaces!")
print(f"  Total configurations to evaluate: {n_chains}")
print(f"  Properties extracted from {full_chain['n_members']} members and {full_chain['n_connectors']} connectors")


Load Transfer System - Using Real Generator Data

1. Generate configurations using actual generators
   Beams: 2 configurations
      W14X68 - tw=0.415", F_y=50.0 ksi
      W14X53 - tw=0.370", F_y=50.0 ksi
   Plates: 12 configurations
      t=[0.25  0.25  0.375 0.375 0.5  ] in, F_y=[36. 36. 36. 36. 36.] ksi
   Bolts: 16 configurations
      First 5: N_r=[3. 3. 4. 4. 3.], size=[0.75 0.75 0.75 0.75 0.75]"
   Welds: 24 configurations
      First 5: size=[0.1875 0.1875 0.1875 0.1875 0.25  ]", F_w=[42. 42. 42. 42. 42.] ksi
   Columns: 1 configurations
      W14X90 - tw=0.440", F_y=50.0 ksi

2. Create connection matrices (relating configs)
   Beam-Plate connections: 384 combinations
   Plate-Column connections: 288 combinations

3. Chain connections (Beam -> Plate -> Column)
   Full chains: 9216 valid combinations
   Chain structure: 3 members, 2 connectors

4. Show first 3 connection chains

   Chain 0:
     Member 0 (Beam)[0]: W14X68
     → Connector 0 (Bolts)[0]: 3.0 rows × 0.75" dia
    

In [12]:
# Debug: Check what's in full_chain
print("Debug full_chain keys:", full_chain.keys())
print("Number of chains:", full_chain['n_chains'])
print("Number of members:", full_chain['n_members'])
print("Number of connectors:", full_chain['n_connectors'])
print("\nFirst chain indices:")
if full_chain['n_chains'] > 0:
    print(f"  member_0_idx[0] = {full_chain['member_0_idx'][0]}")
    print(f"  connector_0_idx[0] = {full_chain['connector_0_idx'][0]}")
    print(f"  member_1_idx[0] = {full_chain['member_1_idx'][0]}")
    print(f"  connector_1_idx[0] = {full_chain['connector_1_idx'][0]}")
    print(f"  member_2_idx[0] = {full_chain['member_2_idx'][0]}")
    print(f"\nColumns array size: {len(columns['designations'])}")
    print(f"Columns: {columns['designations']}")


Debug full_chain keys: dict_keys(['n_chains', 'n_members', 'n_connectors', 'member_0_idx', 'member_1_idx', 'member_2_idx', 'connector_0_idx', 'connector_1_idx', 'member_a_idx', 'member_b_idx', 'member_c_idx', 'connector_2_idx'])
Number of chains: 9216
Number of members: 3
Number of connectors: 2

First chain indices:
  member_0_idx[0] = 0
  connector_0_idx[0] = 0
  member_1_idx[0] = 0
  connector_1_idx[0] = 0
  member_2_idx[0] = 0

Columns array size: 1
Columns: ['W14X90']


In [13]:
# NEW FORMAT: Pass all data in order (member0, connector0, member1, connector1, member2)
props = extract_connection_properties(
    full_chain,
    beams, bolts, plates, welds, columns,  # All 5 data dicts in order!
    properties_to_extract={
        'member_0': ['tw', 'F_y', 'F_u'],           # Beam
        'connector_0': ['bolt_size', 'N_r', 'F_nv'], # Bolts
        'member_1': ['t', 'F_y', 'F_u'],             # Plate
        'connector_1': ['weld_size', 'F_w'],         # Welds
        'member_2': ['tw', 'F_y', 'F_u']             # Column
    }
)

# Properties are returned with prefixes: m0_, c0_, m1_, c1_, m2_
print("Extracted properties (first 3 chains):")
print(f"  Beam (m0) tw: {props['m0_tw'][:3]}")
print(f"  Bolts (c0) N_r: {props['c0_N_r'][:3]}")
print(f"  Plate (m1) t: {props['m1_t'][:3]}")
print(f"  Welds (c1) size: {props['c1_weld_size'][:3]}")
print(f"  Column (m2) tw: {props['m2_tw'][:3]}")

Extracted properties (first 3 chains):
  Beam (m0) tw: [0.415 0.415 0.415]
  Bolts (c0) N_r: [3. 3. 3.]
  Plate (m1) t: [0.25 0.25 0.25]
  Welds (c1) size: [0.1875 0.1875 0.1875]
  Column (m2) tw: [0.44 0.44 0.44]


In [15]:
full_chain

{'n_chains': 9216,
 'n_members': 3,
 'n_connectors': 2,
 'member_0_idx': array([0, 0, 0, ..., 1, 1, 1], shape=(9216,), dtype=int32),
 'member_1_idx': array([ 0,  0,  0, ..., 11, 11, 11], shape=(9216,), dtype=int32),
 'member_2_idx': array([0, 0, 0, ..., 0, 0, 0], shape=(9216,), dtype=int32),
 'connector_0_idx': array([ 0,  0,  0, ..., 15, 15, 15], shape=(9216,), dtype=int32),
 'connector_1_idx': array([ 0,  0,  0, ..., 15, 15, 15], shape=(9216,), dtype=int32),
 'member_a_idx': array([0, 0, 0, ..., 1, 1, 1], shape=(9216,), dtype=int32),
 'member_b_idx': array([ 0,  0,  0, ..., 11, 11, 11], shape=(9216,), dtype=int32),
 'member_c_idx': array([0, 0, 0, ..., 0, 0, 0], shape=(9216,), dtype=int32),
 'connector_2_idx': array([ 0,  0,  0, ..., 15, 15, 15], shape=(9216,), dtype=int32)}

In [16]:
len(bolts['N_r'])

16

## Multi-Interface Chaining Example

Demonstration of chaining 3+ connections for complex load paths

In [18]:
# Example: Chain 3 connections - Beam -> Bolts -> Plate1 -> Welds -> Plate2 -> Bolts -> Column
# This creates a 4-member, 3-connector chain

print("Multi-Connection Chain Example")
print("=" * 70)

# Generate second set of plates (could be gusset or stiffener)
plates2 = generate_shear_plates(
    plate_grade_id=[1],  # A572_50
    t=[0.5, 0.625],
    a=[2]
)
print(f"Plates2: {len(plates2['t'])} configurations")

# Create Connection 1: Beam -> Bolts -> Plate1
conn1 = generate_connection_combinations(
    member_a_count=len(beams['designations']),
    member_b_count=len(plates['t']),
    connector_count=len(bolts['bolt_size'])
)
print(f"Connection 1 (Beam-Plate1): {conn1['n_connections']} combinations")

# Create Connection 2: Plate1 -> Welds -> Plate2
conn2 = generate_connection_combinations(
    member_a_count=len(plates['t']),
    member_b_count=len(plates2['t']),
    connector_count=len(welds['weld_size'])
)
print(f"Connection 2 (Plate1-Plate2): {conn2['n_connections']} combinations")

# Create Connection 3: Plate2 -> Bolts -> Column
conn3 = generate_connection_combinations(
    member_a_count=len(plates2['t']),
    member_b_count=len(columns['designations']),
    connector_count=len(bolts['bolt_size'])
)
print(f"Connection 3 (Plate2-Column): {conn3['n_connections']} combinations")

# Chain all 3 connections!
complex_chain = chain_connections(conn1, conn2, conn3)
print(f"\nChained all 3 connections:")
print(f"  Total valid chains: {complex_chain['n_chains']}")
print(f"  Members: {complex_chain['n_members']}")
print(f"  Connectors: {complex_chain['n_connectors']}")

# Show structure of first chain
if complex_chain['n_chains'] > 0:
    print(f"\nFirst chain structure:")
    print(f"  Member 0: Beam[{complex_chain['member_0_idx'][0]}]")
    print(f"  → Connector 0: Bolt[{complex_chain['connector_0_idx'][0]}]")
    print(f"  → Member 1: Plate1[{complex_chain['member_1_idx'][0]}]")
    print(f"  → Connector 1: Weld[{complex_chain['connector_1_idx'][0]}]")
    print(f"  → Member 2: Plate2[{complex_chain['member_2_idx'][0]}]")
    print(f"  → Connector 2: Bolt[{complex_chain['connector_2_idx'][0]}]")
    print(f"  → Member 3: Column[{complex_chain['member_3_idx'][0]}]")

# Extract properties from all interfaces
multi_props = extract_connection_properties(
    complex_chain,
    beams, bolts, plates, welds, plates2, bolts, columns,  # 4 members, 3 connectors
    properties_to_extract={
        'member_0': ['designations', 'tw'],
        'connector_0': ['N_r'],
        'member_1': ['t'],
        'connector_1': ['weld_size'],
        'member_2': ['t'],
        'connector_2': ['N_r'],
        'member_3': ['designations', 'tw']
    }
)

print(f"\nExtracted properties from all 7 interfaces:")
print(f"  Beam designation: {multi_props['m0_designations'][0]}, tw: {multi_props['m0_tw'][0]}")
print(f"  Bolt1 N_r: {multi_props['c0_N_r'][0]}")
print(f"  Plate1 t: {multi_props['m1_t'][0]}")
print(f"  Weld size: {multi_props['c1_weld_size'][0]}")
print(f"  Plate2 t: {multi_props['m2_t'][0]}")
print(f"  Bolt2 N_r: {multi_props['c2_N_r'][0]}")
print(f"  Column designation: {multi_props['m3_designations'][0]}, tw: {multi_props['m3_tw'][0]}")

print("\n" + "=" * 70)
print("✓ Multi-interface chaining working!")
print(f"  System can handle ANY number of connections")
print(f"  All properties accessible for limit state checks")


Multi-Connection Chain Example
Plates2: 2 configurations
Connection 1 (Beam-Plate1): 384 combinations
Connection 2 (Plate1-Plate2): 576 combinations
Connection 3 (Plate2-Column): 32 combinations

Chained all 3 connections:
  Total valid chains: 294912
  Members: 4
  Connectors: 3

First chain structure:
  Member 0: Beam[0]
  → Connector 0: Bolt[0]
  → Member 1: Plate1[0]
  → Connector 1: Weld[0]
  → Member 2: Plate2[0]
  → Connector 2: Bolt[0]
  → Member 3: Column[0]

Extracted properties from all 7 interfaces:
  Beam designation: W14X68, tw: 0.41499999165534973
  Bolt1 N_r: 3.0
  Plate1 t: 0.25
  Weld size: 0.1875
  Plate2 t: 0.5
  Bolt2 N_r: 3.0
  Column designation: W14X90, tw: 0.4399999976158142

✓ Multi-interface chaining working!
  System can handle ANY number of connections
  All properties accessible for limit state checks


In [19]:
complex_chain = chain_connections(conn1)
# Extract properties from all interfaces
multi_props = extract_connection_properties(
    complex_chain,
    beams, bolts, plates,  # 4 members, 3 connectors
    properties_to_extract={
        'member_0': ['designations', 't_w'],
        'connector_0': ['N_r'],
        'member_1': ['t'],

    }
)


In [20]:
multi_props

{'m0_designations': array(['W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68',
        'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68',
        'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68',
        'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68',
        'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68',
        'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68',
        'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68',
        'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68',
        'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68',
        'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68',
        'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68',
        'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68',
        'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68',
        'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68', 'W14X68',
        'W14X68', 'W14X68', '